# 🕋 9,000 Multi-Scholar Fatwa Dataset: Generation & GPT-4o Mini Labeling
### Sampling 6,000 New Candidates + Merging with Existing 3,000 + Zero-Waste Labeling

This notebook generates and labels the full **9,000 Fatwa Relevance Dataset**:
1. **Candidate Retrieval (6,000 items)**: Retrieves and Mizan-reranks candidates for **500 new diverse jurisprudential questions** across 20 categories.
2. **Quota per Question (12 candidates/Q)**:
   - Sheikh Ibn Baz: 4 candidates
   - Sheikh Ibn Uthaymeen: 2 candidates
   - Al-Lajnah Ad-Da'imah: 2 candidates
   - Sheikh Saleh Al-Fawzan: 1 candidate
   - Sheikh Abdullah al-Manee': 1 candidate
   - Sheikh Abdulaziz Aal ash-Sheikh: 1 candidate
   - Sheikh Saleh bin Humaid: 1 candidate
   - *Total: 12 × 500 = 6,000 new candidates (no backup needed).*
3. **Smart Ingestion of Existing 3,000 Dataset**: Ingests your previously labeled 3,000 dataset and merges it.
4. **Targeted Labeling**: Sends **ONLY the 6,000 new candidates** to GPT-4o Mini with multi-threading & auto-checkpointing ($0 re-billed for the existing 3,000).
5. **Output**: Exports `dataset_9k_labeled.json`, `dataset_9k_labeled.csv`, and `dataset_9k_bundle.zip`.

---

### Hardware Requirements
- **Accelerator**: GPU T4 × 1 or T4 × 2 / P100.
- **Internet**: **ON** (Right sidebar -> Settings -> Internet -> On).


In [ ]:
# 1. Install & Verify Dependencies
!pip install -q sentence-transformers python-docx

import os
import sys
import time
import json
import zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

print("PyTorch Version:         ", torch.__version__)
print("CUDA Available:          ", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:              ", torch.cuda.get_device_name(0))


In [ ]:
# 2. Download Multi-Scholar Corpus & Precomputed Embeddings from Hugging Face
from huggingface_hub import hf_hub_download

# Optional token from Kaggle Secrets if present
token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    pass
if not token:
    token = os.getenv('HF_TOKEN')

DATA_REPO_ID = 'ZenoAbderrahim/binbaz_fatwas_rag'
DATA_DIR = Path('/kaggle/working/data') if Path('/kaggle/working').exists() else Path('data')
DATA_DIR.mkdir(parents=True, exist_ok=True)

FILES_TO_DOWNLOAD = [
    'multi_scholar_clean.csv',
    'qwen3_06b_multi_scholar_embeddings.pt',
]

print(f"Syncing corpus files from Hugging Face dataset: '{DATA_REPO_ID}'...")
for fname in FILES_TO_DOWNLOAD:
    dest = DATA_DIR / fname
    if not dest.exists():
        print(f"  • Downloading {fname} ...")
        hf_hub_download(repo_id=DATA_REPO_ID, repo_type='dataset', filename=fname, local_dir=DATA_DIR, token=token)
    else:
        print(f"  ✓ {fname} already present.")

print("\nLoading cleaned fatwa corpus...")
corpus_df = pd.read_csv(DATA_DIR / 'multi_scholar_clean.csv')
print(f"✓ Total corpus size: {len(corpus_df):,} fatwas across {corpus_df['source'].nunique()} scholars:")
print(corpus_df['source'].value_counts().to_string())

print("\nLoading precomputed Qwen 0.6B corpus embeddings...")
corpus_embeddings = torch.load(DATA_DIR / 'qwen3_06b_multi_scholar_embeddings.pt', map_location='cpu', weights_only=False)
print(f"✓ Corpus embeddings shape: {corpus_embeddings.shape}")


In [ ]:
# 3. Load the 500 New Jurisprudential Questions
# Embedded directly to guarantee 100% fidelity (IDs 251..750)
QUESTIONS_500 = [
  {
    "question_id": 251,
    "category": "الجنائز والقبور",
    "question": "ما حكم تلقين الميت الشهادة عند الاحتضار؟"
  },
  {
    "question_id": 252,
    "category": "الجنائز والقبور",
    "question": "ماذا يشرع للمسلم فعله عند حضور وفاة شخص؟"
  },
  {
    "question_id": 253,
    "category": "الجنائز والقبور",
    "question": "ما حكم توجيه المحتضر إلى القبلة؟"
  },
  {
    "question_id": 254,
    "category": "الجنائز والقبور",
    "question": "ما حكم تغميض عيني الميت بعد وفاته؟"
  },
  {
    "question_id": 255,
    "category": "الجنائز والقبور",
    "question": "هل يجوز كشف وجه الميت وتقبيله بعد موته؟"
  },
  {
    "question_id": 256,
    "category": "الجنائز والقبور",
    "question": "ما حكم الإسراع بتجهيز الميت ودفنه؟"
  },
  {
    "question_id": 257,
    "category": "الجنائز والقبور",
    "question": "ما صفة غسل الميت المسلم؟"
  },
  {
    "question_id": 258,
    "category": "الجنائز والقبور",
    "question": "هل يجوز أن يغسل الرجل زوجته بعد وفاتها؟"
  },
  {
    "question_id": 259,
    "category": "الجنائز والقبور",
    "question": "ما حكم تكفين الميت في أكثر من ثلاثة أثواب؟"
  },
  {
    "question_id": 260,
    "category": "الجنائز والقبور",
    "question": "ما حكم الصلاة على الجنازة في المقبرة؟"
  },
  {
    "question_id": 261,
    "category": "الجنائز والقبور",
    "question": "هل يجوز تكرار صلاة الجنازة على الميت؟"
  },
  {
    "question_id": 262,
    "category": "الجنائز والقبور",
    "question": "ما حكم صلاة الغائب على مسلم مات في بلد بعيد؟"
  },
  {
    "question_id": 263,
    "category": "الجنائز والقبور",
    "question": "ما الذي يقال في صلاة الجنازة؟"
  },
  {
    "question_id": 264,
    "category": "الجنائز والقبور",
    "question": "هل يشرع رفع اليدين في كل تكبيرة من تكبيرات الجنازة؟"
  },
  {
    "question_id": 265,
    "category": "الجنائز والقبور",
    "question": "ما حكم اتباع النساء للجنائز إلى المقبرة؟"
  },
  {
    "question_id": 266,
    "category": "الجنائز والقبور",
    "question": "ما حكم الجلوس قبل وضع الجنازة عن الأعناق؟"
  },
  {
    "question_id": 267,
    "category": "الجنائز والقبور",
    "question": "هل يجوز دفن أكثر من ميت في قبر واحد عند عدم الضرورة؟"
  },
  {
    "question_id": 268,
    "category": "الجنائز والقبور",
    "question": "ما حكم البناء على القبور أو تجصيصها؟"
  },
  {
    "question_id": 269,
    "category": "الجنائز والقبور",
    "question": "هل يجوز وضع علامة بسيطة لمعرفة القبر؟"
  },
  {
    "question_id": 270,
    "category": "الجنائز والقبور",
    "question": "ما حكم زيارة المقابر في يوم مخصوص من كل أسبوع؟"
  },
  {
    "question_id": 271,
    "category": "الجنائز والقبور",
    "question": "هل ينتفع الميت بدعاء أقاربه بعد دفنه؟"
  },
  {
    "question_id": 272,
    "category": "الجنائز والقبور",
    "question": "ما حكم إهداء ثواب قراءة القرآن للميت؟"
  },
  {
    "question_id": 273,
    "category": "الجنائز والقبور",
    "question": "ما حكم إقامة وليمة بعد وفاة أحد الأقارب؟"
  },
  {
    "question_id": 274,
    "category": "الجنائز والقبور",
    "question": "هل يجوز صنع الطعام لأهل الميت في أيام العزاء؟"
  },
  {
    "question_id": 275,
    "category": "الجنائز والقبور",
    "question": "ما حكم تخصيص أيام محددة للعزاء والاجتماع فيها؟"
  },
  {
    "question_id": 276,
    "category": "الأيمان والنذور والكفارات",
    "question": "ما حكم من حلف على أمر ثم ظهر له أن غيره خير منه؟"
  },
  {
    "question_id": 277,
    "category": "الأيمان والنذور والكفارات",
    "question": "ما كفارة اليمين إذا حنث المسلم في يمينه؟"
  },
  {
    "question_id": 278,
    "category": "الأيمان والنذور والكفارات",
    "question": "هل تنعقد اليمين إذا قالها الإنسان دون قصد الحلف؟"
  },
  {
    "question_id": 279,
    "category": "الأيمان والنذور والكفارات",
    "question": "ما حكم الحلف كذبًا لإثبات حق ضائع؟"
  },
  {
    "question_id": 280,
    "category": "الأيمان والنذور والكفارات",
    "question": "هل يجوز الحلف بالمصحف؟"
  },
  {
    "question_id": 281,
    "category": "الأيمان والنذور والكفارات",
    "question": "ما حكم قول الإنسان: عليّ عهد الله أن أفعل كذا؟"
  },
  {
    "question_id": 282,
    "category": "الأيمان والنذور والكفارات",
    "question": "ما الفرق بين اليمين والنذر؟"
  },
  {
    "question_id": 283,
    "category": "الأيمان والنذور والكفارات",
    "question": "ما حكم النذر المعلق على حصول أمر دنيوي؟"
  },
  {
    "question_id": 284,
    "category": "الأيمان والنذور والكفارات",
    "question": "هل يجب الوفاء بنذر الطاعة إذا عجز صاحبه عنه؟"
  },
  {
    "question_id": 285,
    "category": "الأيمان والنذور والكفارات",
    "question": "ما حكم النذر بمعصية؟"
  },
  {
    "question_id": 286,
    "category": "الأيمان والنذور والكفارات",
    "question": "ماذا يلزم من نذر صيام أيام ثم مرض ولم يستطع؟"
  },
  {
    "question_id": 287,
    "category": "الأيمان والنذور والكفارات",
    "question": "ما حكم من نذر ذبح شاة ولم يحدد جهة صرفها؟"
  },
  {
    "question_id": 288,
    "category": "الأيمان والنذور والكفارات",
    "question": "هل يجوز تحويل النذر من عمل معين إلى عمل آخر؟"
  },
  {
    "question_id": 289,
    "category": "الأيمان والنذور والكفارات",
    "question": "ما حكم من نذر صدقة ثم احتاج إلى المال؟"
  },
  {
    "question_id": 290,
    "category": "الأيمان والنذور والكفارات",
    "question": "هل تجب كفارة على من حلف بالطلاق وهو لا يريد وقوعه؟"
  },
  {
    "question_id": 291,
    "category": "الأيمان والنذور والكفارات",
    "question": "ما حكم كثرة الحلف في البيع والشراء؟"
  },
  {
    "question_id": 292,
    "category": "الأيمان والنذور والكفارات",
    "question": "ما حكم قول: أقسم بالله، على سبيل العادة دون نية؟"
  },
  {
    "question_id": 293,
    "category": "الأيمان والنذور والكفارات",
    "question": "هل تجب كفارة اليمين على من حنث ناسيًا؟"
  },
  {
    "question_id": 294,
    "category": "الأيمان والنذور والكفارات",
    "question": "كيف يطعم المسلم المساكين في كفارة اليمين؟"
  },
  {
    "question_id": 295,
    "category": "الأيمان والنذور والكفارات",
    "question": "هل يجوز إخراج قيمة الطعام نقدًا في كفارة اليمين؟"
  },
  {
    "question_id": 296,
    "category": "الأيمان والنذور والكفارات",
    "question": "ما حكم من كرر اليمين على الفعل نفسه ثم حنث؟"
  },
  {
    "question_id": 297,
    "category": "الأيمان والنذور والكفارات",
    "question": "ما حكم من حلف ألا يكلم قريبه ثم أراد صلة رحمه؟"
  },
  {
    "question_id": 298,
    "category": "الأيمان والنذور والكفارات",
    "question": "هل يلزم الوفاء بوعد اقترن بيمين؟"
  },
  {
    "question_id": 299,
    "category": "الأيمان والنذور والكفارات",
    "question": "ما حكم النذر لغير الله؟"
  },
  {
    "question_id": 300,
    "category": "الأيمان والنذور والكفارات",
    "question": "متى تكون كفارة النذر كفارة يمين؟"
  },
  {
    "question_id": 301,
    "category": "الذكر والدعاء والرقائق",
    "question": "ما حكم الالتزام بعدد معين من الأذكار دون دليل؟"
  },
  {
    "question_id": 302,
    "category": "الذكر والدعاء والرقائق",
    "question": "هل يجوز الدعاء بصيغة لم ترد في السنة إذا كان معناها صحيحًا؟"
  },
  {
    "question_id": 303,
    "category": "الذكر والدعاء والرقائق",
    "question": "ما حكم تخصيص دعاء معين بعد كل صلاة مفروضة؟"
  },
  {
    "question_id": 304,
    "category": "الذكر والدعاء والرقائق",
    "question": "هل يشرع رفع اليدين في كل دعاء؟"
  },
  {
    "question_id": 305,
    "category": "الذكر والدعاء والرقائق",
    "question": "ما حكم مسح الوجه باليدين بعد الدعاء؟"
  },
  {
    "question_id": 306,
    "category": "الذكر والدعاء والرقائق",
    "question": "ما حكم الاجتماع على الذكر بصوت واحد بعد الصلاة؟"
  },
  {
    "question_id": 307,
    "category": "الذكر والدعاء والرقائق",
    "question": "هل يجوز قراءة أذكار الصباح قبل طلوع الفجر؟"
  },
  {
    "question_id": 308,
    "category": "الذكر والدعاء والرقائق",
    "question": "متى يبدأ وقت أذكار المساء وينتهي؟"
  },
  {
    "question_id": 309,
    "category": "الذكر والدعاء والرقائق",
    "question": "ما حكم ترك أذكار النوم بسبب التعب؟"
  },
  {
    "question_id": 310,
    "category": "الذكر والدعاء والرقائق",
    "question": "هل يشرع قول البسملة عند دخول المنزل؟"
  },
  {
    "question_id": 311,
    "category": "الذكر والدعاء والرقائق",
    "question": "ما الذكر المشروع عند دخول الخلاء والخروج منه؟"
  },
  {
    "question_id": 312,
    "category": "الذكر والدعاء والرقائق",
    "question": "ما حكم ذكر الله في الحمام إذا كان موضع الاستحمام منفصلًا عن موضع قضاء الحاجة؟"
  },
  {
    "question_id": 313,
    "category": "الذكر والدعاء والرقائق",
    "question": "هل يجوز قراءة القرآن من الهاتف دون وضوء؟"
  },
  {
    "question_id": 314,
    "category": "الذكر والدعاء والرقائق",
    "question": "ما حكم تخصيص سورة معينة لتيسير الزواج دون دليل؟"
  },
  {
    "question_id": 315,
    "category": "الذكر والدعاء والرقائق",
    "question": "هل ورد دعاء مخصوص عند لبس الثوب الجديد؟"
  },
  {
    "question_id": 316,
    "category": "الذكر والدعاء والرقائق",
    "question": "ما حكم تعليق ورقة فيها أذكار على جدار المنزل؟"
  },
  {
    "question_id": 317,
    "category": "الذكر والدعاء والرقائق",
    "question": "هل يجوز الدعاء للميت عند المرور بقبره؟"
  },
  {
    "question_id": 318,
    "category": "الذكر والدعاء والرقائق",
    "question": "ما حكم الدعاء على المسلم بسبب خلاف شخصي؟"
  },
  {
    "question_id": 319,
    "category": "الذكر والدعاء والرقائق",
    "question": "هل يجوز الدعاء بطلب أمر مستحيل عادة؟"
  },
  {
    "question_id": 320,
    "category": "الذكر والدعاء والرقائق",
    "question": "ما حكم الاعتداء في الدعاء؟"
  },
  {
    "question_id": 321,
    "category": "الذكر والدعاء والرقائق",
    "question": "هل يشرع الإكثار من الصلاة على النبي يوم الجمعة؟"
  },
  {
    "question_id": 322,
    "category": "الذكر والدعاء والرقائق",
    "question": "ما حكم استعمال السبحة للعد مع وجود الأصابع؟"
  },
  {
    "question_id": 323,
    "category": "الذكر والدعاء والرقائق",
    "question": "هل يجوز الذكر بالقلب دون تحريك اللسان؟"
  },
  {
    "question_id": 324,
    "category": "الذكر والدعاء والرقائق",
    "question": "ما حكم قراءة القرآن بنية الرقية على ماء ثم شربه؟"
  },
  {
    "question_id": 325,
    "category": "الذكر والدعاء والرقائق",
    "question": "هل يجوز أن يطلب المسلم من غيره أن يرقيه؟"
  },
  {
    "question_id": 326,
    "category": "العلم وطلبه وآدابه",
    "question": "ما حكم البدء بطلب العلم قبل حفظ القرآن؟"
  },
  {
    "question_id": 327,
    "category": "العلم وطلبه وآدابه",
    "question": "كيف يرتب طالب العلم المبتدئ دراسة العقيدة والفقه والحديث؟"
  },
  {
    "question_id": 328,
    "category": "العلم وطلبه وآدابه",
    "question": "ما حكم قراءة كتب الخلاف قبل التأهل العلمي؟"
  },
  {
    "question_id": 329,
    "category": "العلم وطلبه وآدابه",
    "question": "هل يجوز لطالب العلم أن يفتي الناس بما قرأه في كتاب؟"
  },
  {
    "question_id": 330,
    "category": "العلم وطلبه وآدابه",
    "question": "ما حكم نقل فتوى عالم دون التأكد من صحة نسبتها إليه؟"
  },
  {
    "question_id": 331,
    "category": "العلم وطلبه وآدابه",
    "question": "كيف يتعامل طالب العلم مع مسألة لا يعرف حكمها؟"
  },
  {
    "question_id": 332,
    "category": "العلم وطلبه وآدابه",
    "question": "ما حكم مناظرة المبتدئ لأهل العلم في المسائل الدقيقة؟"
  },
  {
    "question_id": 333,
    "category": "العلم وطلبه وآدابه",
    "question": "هل يجوز لطالب العلم أن يترك الدراسة النظامية للتفرغ للعلم الشرعي؟"
  },
  {
    "question_id": 334,
    "category": "العلم وطلبه وآدابه",
    "question": "ما حكم طلب العلم عبر الدروس المسجلة فقط؟"
  },
  {
    "question_id": 335,
    "category": "العلم وطلبه وآدابه",
    "question": "هل يشترط فهم العربية لفهم كتب العقيدة والحديث؟"
  },
  {
    "question_id": 336,
    "category": "العلم وطلبه وآدابه",
    "question": "ما حكم التعصب لشيخ معين ورفض كلام غيره من العلماء؟"
  },
  {
    "question_id": 337,
    "category": "العلم وطلبه وآدابه",
    "question": "كيف يتصرف طالب العلم إذا ظهر له خطأ في كلام شيخه؟"
  },
  {
    "question_id": 338,
    "category": "العلم وطلبه وآدابه",
    "question": "ما حكم نشر مقطع علمي قبل التأكد من سياقه؟"
  },
  {
    "question_id": 339,
    "category": "العلم وطلبه وآدابه",
    "question": "هل يجوز اختصار كلام العالم بما يغير المعنى؟"
  },
  {
    "question_id": 340,
    "category": "العلم وطلبه وآدابه",
    "question": "ما حكم أخذ العلم عن شخص مجهول المنهج؟"
  },
  {
    "question_id": 341,
    "category": "العلم وطلبه وآدابه",
    "question": "كيف يميز طالب العلم بين النصيحة والتشهير؟"
  },
  {
    "question_id": 342,
    "category": "العلم وطلبه وآدابه",
    "question": "ما حكم تصدر الشخص للإجابة عن الأسئلة الشرعية عبر الإنترنت؟"
  },
  {
    "question_id": 343,
    "category": "العلم وطلبه وآدابه",
    "question": "هل يجوز سؤال أكثر من عالم في المسألة نفسها بحثًا عن الدليل؟"
  },
  {
    "question_id": 344,
    "category": "العلم وطلبه وآدابه",
    "question": "ما حكم تتبع الرخص بين أقوال العلماء؟"
  },
  {
    "question_id": 345,
    "category": "العلم وطلبه وآدابه",
    "question": "كيف يتعامل المسلم مع المسائل الاجتهادية التي اختلف فيها العلماء؟"
  },
  {
    "question_id": 346,
    "category": "العلم وطلبه وآدابه",
    "question": "ما حكم حفظ المتون دون فهم معانيها؟"
  },
  {
    "question_id": 347,
    "category": "العلم وطلبه وآدابه",
    "question": "هل يجوز لطالب العلم أن يعلّم غيره ما لم يتقنه تمامًا؟"
  },
  {
    "question_id": 348,
    "category": "العلم وطلبه وآدابه",
    "question": "ما حكم الاستشهاد بحديث لم يتأكد المرء من درجته؟"
  },
  {
    "question_id": 349,
    "category": "العلم وطلبه وآدابه",
    "question": "كيف يتثبت المسلم من صحة كتاب منسوب إلى عالم؟"
  },
  {
    "question_id": 350,
    "category": "العلم وطلبه وآدابه",
    "question": "ما حكم ترك العمل بالعلم خوفًا من الرياء؟"
  },
  {
    "question_id": 351,
    "category": "الدعوة والإنكار وآداب الخلاف",
    "question": "ما ضوابط إنكار المنكر إذا خشي المسلم وقوع مفسدة أكبر؟"
  },
  {
    "question_id": 352,
    "category": "الدعوة والإنكار وآداب الخلاف",
    "question": "هل يجب إنكار المنكر على كل حال أم تراعى القدرة؟"
  },
  {
    "question_id": 353,
    "category": "الدعوة والإنكار وآداب الخلاف",
    "question": "ما حكم نصح شخص على الملأ أمام الناس؟"
  },
  {
    "question_id": 354,
    "category": "الدعوة والإنكار وآداب الخلاف",
    "question": "كيف ينصح المسلم قريبًا يصر على معصية؟"
  },
  {
    "question_id": 355,
    "category": "الدعوة والإنكار وآداب الخلاف",
    "question": "ما حكم هجر المسلم لأجل بدعة يراها فيه؟"
  },
  {
    "question_id": 356,
    "category": "الدعوة والإنكار وآداب الخلاف",
    "question": "متى يكون هجر العاصي مشروعًا ومتى لا يكون كذلك؟"
  },
  {
    "question_id": 357,
    "category": "الدعوة والإنكار وآداب الخلاف",
    "question": "ما حكم نشر أخطاء الدعاة للتحذير منهم؟"
  },
  {
    "question_id": 358,
    "category": "الدعوة والإنكار وآداب الخلاف",
    "question": "ما الضوابط الشرعية للتحذير من شخص بعينه؟"
  },
  {
    "question_id": 359,
    "category": "الدعوة والإنكار وآداب الخلاف",
    "question": "هل يجوز الرد على المخالف إذا كان الرد سيؤدي إلى فتنة؟"
  },
  {
    "question_id": 360,
    "category": "الدعوة والإنكار وآداب الخلاف",
    "question": "ما حكم وصف كل مخالف في مسألة اجتهادية بأنه مبتدع؟"
  },
  {
    "question_id": 361,
    "category": "الدعوة والإنكار وآداب الخلاف",
    "question": "كيف يتعامل المسلم مع من يسيء إلى عالم من العلماء؟"
  },
  {
    "question_id": 362,
    "category": "الدعوة والإنكار وآداب الخلاف",
    "question": "ما حكم نصح الوالدين إذا ارتكبا منكرًا؟"
  },
  {
    "question_id": 363,
    "category": "الدعوة والإنكار وآداب الخلاف",
    "question": "هل يجوز إنكار المنكر على المسؤول في العمل أمام زملائه؟"
  },
  {
    "question_id": 364,
    "category": "الدعوة والإنكار وآداب الخلاف",
    "question": "ما حكم الدعوة إلى الله باستخدام القصص غير الثابتة؟"
  },
  {
    "question_id": 365,
    "category": "الدعوة والإنكار وآداب الخلاف",
    "question": "هل يجوز نشر أحاديث ضعيفة في الترغيب والترهيب؟"
  },
  {
    "question_id": 366,
    "category": "الدعوة والإنكار وآداب الخلاف",
    "question": "ما حكم استخدام عبارات قاسية في نصيحة العاصي؟"
  },
  {
    "question_id": 367,
    "category": "الدعوة والإنكار وآداب الخلاف",
    "question": "كيف يوازن الداعية بين الرفق وبيان الحق؟"
  },
  {
    "question_id": 368,
    "category": "الدعوة والإنكار وآداب الخلاف",
    "question": "ما حكم ترك نصح شخص خوفًا من خسارة صداقته؟"
  },
  {
    "question_id": 369,
    "category": "الدعوة والإنكار وآداب الخلاف",
    "question": "هل يجوز للمسلم أن يرد السلام على من هجره؟"
  },
  {
    "question_id": 370,
    "category": "الدعوة والإنكار وآداب الخلاف",
    "question": "ما حكم مقاطعة قريب بسبب خلاف في مسألة دينية؟"
  },
  {
    "question_id": 371,
    "category": "الدعوة والإنكار وآداب الخلاف",
    "question": "كيف يتعامل المسلم مع خلاف العلماء دون انتقاص أحدهم؟"
  },
  {
    "question_id": 372,
    "category": "الدعوة والإنكار وآداب الخلاف",
    "question": "ما حكم جعل الولاء والبراء متعلقًا بالمواقف الشخصية؟"
  },
  {
    "question_id": 373,
    "category": "الدعوة والإنكار وآداب الخلاف",
    "question": "هل يجوز نقل كلام شخص في مجلس خاص للتحذير منه؟"
  },
  {
    "question_id": 374,
    "category": "الدعوة والإنكار وآداب الخلاف",
    "question": "ما حكم نشر مقاطع الردود دون نشر كلام الطرف الآخر كاملًا؟"
  },
  {
    "question_id": 375,
    "category": "الدعوة والإنكار وآداب الخلاف",
    "question": "ما واجب المسلم إذا سمع شبهة ولم يعرف جوابها؟"
  },
  {
    "question_id": 376,
    "category": "القرآن والتلاوة والحفظ",
    "question": "ما حكم قراءة القرآن من المصحف دون تحريك الشفتين؟"
  },
  {
    "question_id": 377,
    "category": "القرآن والتلاوة والحفظ",
    "question": "هل يشترط الوضوء لمس المصحف الإلكتروني؟"
  },
  {
    "question_id": 378,
    "category": "القرآن والتلاوة والحفظ",
    "question": "ما حكم قراءة القرآن في مكان العمل أثناء وقت الدوام؟"
  },
  {
    "question_id": 379,
    "category": "القرآن والتلاوة والحفظ",
    "question": "هل يجوز قطع تلاوة القرآن للرد على الهاتف؟"
  },
  {
    "question_id": 380,
    "category": "القرآن والتلاوة والحفظ",
    "question": "ما حكم قراءة القرآن بسرعة شديدة بقصد ختمه؟"
  },
  {
    "question_id": 381,
    "category": "القرآن والتلاوة والحفظ",
    "question": "ما الحد الذي ينبغي مراعاته في أحكام التجويد؟"
  },
  {
    "question_id": 382,
    "category": "القرآن والتلاوة والحفظ",
    "question": "هل يأثم من يخطئ في التلاوة بسبب ضعف لغته؟"
  },
  {
    "question_id": 383,
    "category": "القرآن والتلاوة والحفظ",
    "question": "ما حكم تصحيح الإمام في القراءة أثناء الصلاة؟"
  },
  {
    "question_id": 384,
    "category": "القرآن والتلاوة والحفظ",
    "question": "هل يجوز الاقتداء بقارئ من تسجيل صوتي في الصلاة؟"
  },
  {
    "question_id": 385,
    "category": "القرآن والتلاوة والحفظ",
    "question": "ما حكم تخصيص وقت يومي للحفظ دون مراجعة المحفوظ القديم؟"
  },
  {
    "question_id": 386,
    "category": "القرآن والتلاوة والحفظ",
    "question": "كيف يتعامل حافظ القرآن مع كثرة النسيان؟"
  },
  {
    "question_id": 387,
    "category": "القرآن والتلاوة والحفظ",
    "question": "ما حكم نسيان آيات سبق حفظها؟"
  },
  {
    "question_id": 388,
    "category": "القرآن والتلاوة والحفظ",
    "question": "هل يجوز قراءة القرآن بنية الرقية وبنية التعبد معًا؟"
  },
  {
    "question_id": 389,
    "category": "القرآن والتلاوة والحفظ",
    "question": "ما حكم كتابة آيات قرآنية على لوحة للزينة؟"
  },
  {
    "question_id": 390,
    "category": "القرآن والتلاوة والحفظ",
    "question": "هل يجوز وضع المصحف في حقيبة فيها أشياء أخرى؟"
  },
  {
    "question_id": 391,
    "category": "القرآن والتلاوة والحفظ",
    "question": "ما حكم وضع المصحف في مكان منخفض مع عدم قصد الإهانة؟"
  },
  {
    "question_id": 392,
    "category": "القرآن والتلاوة والحفظ",
    "question": "هل يجوز تعليم الأطفال القرآن مقابل أجرة؟"
  },
  {
    "question_id": 393,
    "category": "القرآن والتلاوة والحفظ",
    "question": "ما حكم أخذ معلم القرآن راتبًا من جمعية خيرية؟"
  },
  {
    "question_id": 394,
    "category": "القرآن والتلاوة والحفظ",
    "question": "هل يجوز تسجيل تلاوة طالب دون إذنه ونشرها؟"
  },
  {
    "question_id": 395,
    "category": "القرآن والتلاوة والحفظ",
    "question": "ما حكم قراءة سورة الكهف ليلة الجمعة بدل يومها؟"
  },
  {
    "question_id": 396,
    "category": "القرآن والتلاوة والحفظ",
    "question": "هل ثبت فضل مخصوص لقراءة سورة معينة بعدد محدد لقضاء الحاجة؟"
  },
  {
    "question_id": 397,
    "category": "القرآن والتلاوة والحفظ",
    "question": "ما حكم قراءة القرآن جماعة بالتناوب بحيث يستمع بعضهم لبعض؟"
  },
  {
    "question_id": 398,
    "category": "القرآن والتلاوة والحفظ",
    "question": "هل يجوز السجود للتلاوة خارج الصلاة دون وضوء؟"
  },
  {
    "question_id": 399,
    "category": "القرآن والتلاوة والحفظ",
    "question": "ما حكم سجود التلاوة لمن يسمع الآية عبر الهاتف؟"
  },
  {
    "question_id": 400,
    "category": "القرآن والتلاوة والحفظ",
    "question": "هل يجوز كتابة آيات من القرآن على ورقة ثم إتلافها بعد الحفظ؟"
  },
  {
    "question_id": 401,
    "category": "الحديث والسنة",
    "question": "كيف يتعامل المسلم مع حديث ضعيف في فضائل الأعمال؟"
  },
  {
    "question_id": 402,
    "category": "الحديث والسنة",
    "question": "ما حكم العمل بحديث لم يعرف درجته؟"
  },
  {
    "question_id": 403,
    "category": "الحديث والسنة",
    "question": "هل يجوز الاحتجاج بحديث مرسل في الأحكام؟"
  },
  {
    "question_id": 404,
    "category": "الحديث والسنة",
    "question": "ما حكم نشر حديث مشهور على وسائل التواصل دون تخريج؟"
  },
  {
    "question_id": 405,
    "category": "الحديث والسنة",
    "question": "كيف يتحقق المسلم من صحة حديث منتشر في الرسائل؟"
  },
  {
    "question_id": 406,
    "category": "الحديث والسنة",
    "question": "ما حكم الاستدلال بحديث فهمه الناقل على غير مراد الشارع؟"
  },
  {
    "question_id": 407,
    "category": "الحديث والسنة",
    "question": "هل يجوز شرح حديث نبوي دون الرجوع إلى كلام أهل العلم؟"
  },
  {
    "question_id": 408,
    "category": "الحديث والسنة",
    "question": "ما حكم رواية الحديث بالمعنى لمن ليس من أهل الاختصاص؟"
  },
  {
    "question_id": 409,
    "category": "الحديث والسنة",
    "question": "هل يلزم ذكر درجة الحديث كلما استشهد به الخطيب؟"
  },
  {
    "question_id": 410,
    "category": "الحديث والسنة",
    "question": "ما حكم نسبة قول إلى النبي صلى الله عليه وسلم مع الشك في ثبوته؟"
  },
  {
    "question_id": 411,
    "category": "الحديث والسنة",
    "question": "كيف يفرق طالب العلم بين الحديث القدسي والقرآن؟"
  },
  {
    "question_id": 412,
    "category": "الحديث والسنة",
    "question": "هل يجوز الاحتجاج بحديث مختلف في صحته دون بيان الخلاف؟"
  },
  {
    "question_id": 413,
    "category": "الحديث والسنة",
    "question": "ما حكم ترك حديث صحيح لأن ظاهره يخالف الرأي؟"
  },
  {
    "question_id": 414,
    "category": "الحديث والسنة",
    "question": "كيف يجمع العلماء بين حديثين يبدو بينهما تعارض؟"
  },
  {
    "question_id": 415,
    "category": "الحديث والسنة",
    "question": "ما حكم العمل بحديث منسوخ لا يعلم المرء نسخه؟"
  },
  {
    "question_id": 416,
    "category": "الحديث والسنة",
    "question": "هل يجوز استخدام تطبيقات تخريج الحديث دون مراجعة المصادر؟"
  },
  {
    "question_id": 417,
    "category": "الحديث والسنة",
    "question": "ما حكم نشر صورة فيها حديث مع خطأ في لفظه؟"
  },
  {
    "question_id": 418,
    "category": "الحديث والسنة",
    "question": "هل يجوز اقتباس جزء من حديث إذا كان حذف بقيته يغير المعنى؟"
  },
  {
    "question_id": 419,
    "category": "الحديث والسنة",
    "question": "ما حكم رواية الأحاديث الإسرائيلية في المواعظ؟"
  },
  {
    "question_id": 420,
    "category": "الحديث والسنة",
    "question": "هل يجوز ذكر قصة دينية لا أصل معروفًا لها للتأثير في الناس؟"
  },
  {
    "question_id": 421,
    "category": "الحديث والسنة",
    "question": "ما ضابط قول العلماء: هذا الحديث لا أصل له؟"
  },
  {
    "question_id": 422,
    "category": "الحديث والسنة",
    "question": "ما حكم الاستشهاد بحديث في مسألة عقدية مع اختلاف العلماء في صحته؟"
  },
  {
    "question_id": 423,
    "category": "الحديث والسنة",
    "question": "هل يجوز ترجمة الحديث إلى لغة أخرى دون ذكر النص العربي؟"
  },
  {
    "question_id": 424,
    "category": "الحديث والسنة",
    "question": "ما حكم تصحيح الأحاديث وتضعيفها لمن لم يدرس علم المصطلح؟"
  },
  {
    "question_id": 425,
    "category": "الحديث والسنة",
    "question": "كيف يتعامل المسلم مع حديث صحيح لم يفهم معناه؟"
  },
  {
    "question_id": 426,
    "category": "المعاملات والحقوق المالية المتنوعة",
    "question": "ما حكم العربون إذا عدل المشتري عن الشراء؟"
  },
  {
    "question_id": 427,
    "category": "المعاملات والحقوق المالية المتنوعة",
    "question": "هل يجوز بيع شيء لا يملكه البائع ثم يشتريه بعد الاتفاق؟"
  },
  {
    "question_id": 428,
    "category": "المعاملات والحقوق المالية المتنوعة",
    "question": "ما حكم بيع السلع قبل قبضها؟"
  },
  {
    "question_id": 429,
    "category": "المعاملات والحقوق المالية المتنوعة",
    "question": "هل يجوز أن يبيع الوكيل السلعة لنفسه؟"
  },
  {
    "question_id": 430,
    "category": "المعاملات والحقوق المالية المتنوعة",
    "question": "ما حكم اشتراط غرامة مالية على التأخر في سداد الدين؟"
  },
  {
    "question_id": 431,
    "category": "المعاملات والحقوق المالية المتنوعة",
    "question": "هل يجوز إسقاط جزء من الدين مقابل تعجيل السداد؟"
  },
  {
    "question_id": 432,
    "category": "المعاملات والحقوق المالية المتنوعة",
    "question": "ما حكم شراء سلعة مع اشتراط هدية من البائع؟"
  },
  {
    "question_id": 433,
    "category": "المعاملات والحقوق المالية المتنوعة",
    "question": "هل يجوز أخذ عمولة مقابل الدلالة على مشترٍ؟"
  },
  {
    "question_id": 434,
    "category": "المعاملات والحقوق المالية المتنوعة",
    "question": "ما حكم الوساطة في بيع عقار مقابل نسبة معلومة؟"
  },
  {
    "question_id": 435,
    "category": "المعاملات والحقوق المالية المتنوعة",
    "question": "هل يجوز أخذ أجر على كتابة عقد بيع بين طرفين؟"
  },
  {
    "question_id": 436,
    "category": "المعاملات والحقوق المالية المتنوعة",
    "question": "ما حكم بيع المنتجات المقلدة إذا كان المشتري يعلم بذلك؟"
  },
  {
    "question_id": 437,
    "category": "المعاملات والحقوق المالية المتنوعة",
    "question": "هل يجوز بيع سلعة مع إخفاء عيب فيها؟"
  },
  {
    "question_id": 438,
    "category": "المعاملات والحقوق المالية المتنوعة",
    "question": "ما حق المشتري إذا اكتشف عيبًا قديمًا بعد الشراء؟"
  },
  {
    "question_id": 439,
    "category": "المعاملات والحقوق المالية المتنوعة",
    "question": "ما حكم البيع في مزاد علني؟"
  },
  {
    "question_id": 440,
    "category": "المعاملات والحقوق المالية المتنوعة",
    "question": "هل يجوز أن يزيد شخص في سعر المزاد وهو لا يريد الشراء؟"
  },
  {
    "question_id": 441,
    "category": "المعاملات والحقوق المالية المتنوعة",
    "question": "ما حكم بيع شيء بسعر مختلف لمن يدفع نقدًا أو بالتقسيط؟"
  },
  {
    "question_id": 442,
    "category": "المعاملات والحقوق المالية المتنوعة",
    "question": "هل يجوز اشتراط دفع عربون غير مسترد في كل الأحوال؟"
  },
  {
    "question_id": 443,
    "category": "المعاملات والحقوق المالية المتنوعة",
    "question": "ما حكم شراء الذهب المستعمل مقابل ذهب جديد مع دفع الفرق؟"
  },
  {
    "question_id": 444,
    "category": "المعاملات والحقوق المالية المتنوعة",
    "question": "هل يجوز استبدال الذهب القديم بذهب جديد مع دفع أجرة الصياغة؟"
  },
  {
    "question_id": 445,
    "category": "المعاملات والحقوق المالية المتنوعة",
    "question": "ما حكم شراء العملات الأجنبية مع تأخير تسليم أحد العوضين؟"
  },
  {
    "question_id": 446,
    "category": "المعاملات والحقوق المالية المتنوعة",
    "question": "هل يجوز إقراض شخص مالًا بشرط أن يقدم خدمة للمقرض؟"
  },
  {
    "question_id": 447,
    "category": "المعاملات والحقوق المالية المتنوعة",
    "question": "ما حكم إعطاء الموظف هدية من عميل له مصلحة عنده؟"
  },
  {
    "question_id": 448,
    "category": "المعاملات والحقوق المالية المتنوعة",
    "question": "هل يجوز للموظف قبول خصم خاص من المورد بسبب منصبه؟"
  },
  {
    "question_id": 449,
    "category": "المعاملات والحقوق المالية المتنوعة",
    "question": "ما حكم استخدام ممتلكات العمل لأغراض شخصية؟"
  },
  {
    "question_id": 450,
    "category": "المعاملات والحقوق المالية المتنوعة",
    "question": "ما حكم أخذ وقت من الدوام لإنجاز عمل خاص؟"
  },
  {
    "question_id": 451,
    "category": "الوقف والوصية والميراث",
    "question": "ما الفرق بين الوقف والصدقة الجارية؟"
  },
  {
    "question_id": 452,
    "category": "الوقف والوصية والميراث",
    "question": "هل يجوز وقف جزء من الراتب شهريًا لمشروع خيري؟"
  },
  {
    "question_id": 453,
    "category": "الوقف والوصية والميراث",
    "question": "ما حكم تغيير شرط الواقف إذا تعذر تنفيذه؟"
  },
  {
    "question_id": 454,
    "category": "الوقف والوصية والميراث",
    "question": "هل يجوز بيع عقار موقوف إذا تعطلت منفعته؟"
  },
  {
    "question_id": 455,
    "category": "الوقف والوصية والميراث",
    "question": "من يتولى إدارة الوقف إذا لم يحدد الواقف ناظرًا؟"
  },
  {
    "question_id": 456,
    "category": "الوقف والوصية والميراث",
    "question": "ما حكم استثمار أموال الوقف في مشروع مباح؟"
  },
  {
    "question_id": 457,
    "category": "الوقف والوصية والميراث",
    "question": "هل يجوز للواقف أن ينتفع بالوقف الذي أنشأه لنفسه؟"
  },
  {
    "question_id": 458,
    "category": "الوقف والوصية والميراث",
    "question": "ما حكم تخصيص الوقف لأحد الأبناء دون بقية الورثة؟"
  },
  {
    "question_id": 459,
    "category": "الوقف والوصية والميراث",
    "question": "ما شروط صحة الوصية؟"
  },
  {
    "question_id": 460,
    "category": "الوقف والوصية والميراث",
    "question": "هل يجوز للإنسان أن يوصي بأكثر من ثلث ماله؟"
  },
  {
    "question_id": 461,
    "category": "الوقف والوصية والميراث",
    "question": "ما حكم الوصية لوارث؟"
  },
  {
    "question_id": 462,
    "category": "الوقف والوصية والميراث",
    "question": "هل تنفذ الوصية إذا لم يشهد عليها أحد؟"
  },
  {
    "question_id": 463,
    "category": "الوقف والوصية والميراث",
    "question": "ما حكم الرجوع عن الوصية في حياة الموصي؟"
  },
  {
    "question_id": 464,
    "category": "الوقف والوصية والميراث",
    "question": "هل يجوز تأخير تقسيم التركة سنوات مع رضا بعض الورثة؟"
  },
  {
    "question_id": 465,
    "category": "الوقف والوصية والميراث",
    "question": "ما حكم منع أحد الورثة من الاطلاع على حسابات التركة؟"
  },
  {
    "question_id": 466,
    "category": "الوقف والوصية والميراث",
    "question": "كيف تسدد ديون الميت قبل تقسيم الميراث؟"
  },
  {
    "question_id": 467,
    "category": "الوقف والوصية والميراث",
    "question": "هل تدخل الهدايا التي أعطاها الأب لبعض أولاده في حساب التركة؟"
  },
  {
    "question_id": 468,
    "category": "الوقف والوصية والميراث",
    "question": "ما حكم تنازل أحد الورثة عن نصيبه قبل وفاة المورث؟"
  },
  {
    "question_id": 469,
    "category": "الوقف والوصية والميراث",
    "question": "هل يجوز توزيع التركة بالتساوي بين الذكور والإناث باتفاق الورثة؟"
  },
  {
    "question_id": 470,
    "category": "الوقف والوصية والميراث",
    "question": "ما حكم بيع نصيب الميراث لأحد الورثة قبل فرز العقار؟"
  },
  {
    "question_id": 471,
    "category": "الوقف والوصية والميراث",
    "question": "هل يرث المسلم قريبه غير المسلم؟"
  },
  {
    "question_id": 472,
    "category": "الوقف والوصية والميراث",
    "question": "ما حكم الميراث إذا مات شخصان في حادث ولم يعرف المتقدم وفاة؟"
  },
  {
    "question_id": 473,
    "category": "الوقف والوصية والميراث",
    "question": "هل يجوز للورثة التصدق عن الميت من أموالهم الخاصة؟"
  },
  {
    "question_id": 474,
    "category": "الوقف والوصية والميراث",
    "question": "ما حكم تخصيص جزء من التركة لتجهيز مجلس عزاء؟"
  },
  {
    "question_id": 475,
    "category": "الوقف والوصية والميراث",
    "question": "كيف يتصرف الورثة إذا كانت التركة تشمل دينًا على أحد الأبناء؟"
  },
  {
    "question_id": 476,
    "category": "الأطعمة والذبائح والصيد",
    "question": "ما حكم أكل اللحوم المستوردة من بلد غير مسلم؟"
  },
  {
    "question_id": 477,
    "category": "الأطعمة والذبائح والصيد",
    "question": "ما الضوابط الشرعية للحكم على ذبيحة مجهولة المصدر؟"
  },
  {
    "question_id": 478,
    "category": "الأطعمة والذبائح والصيد",
    "question": "هل يجوز أكل ذبيحة من لا يصلي؟"
  },
  {
    "question_id": 479,
    "category": "الأطعمة والذبائح والصيد",
    "question": "ما حكم أكل الطعام الذي خالطه شيء محرم بكمية يسيرة؟"
  },
  {
    "question_id": 480,
    "category": "الأطعمة والذبائح والصيد",
    "question": "هل يجوز تناول طعام طبخ في إناء استعمل سابقًا في محرم؟"
  },
  {
    "question_id": 481,
    "category": "الأطعمة والذبائح والصيد",
    "question": "ما حكم أكل الجيلاتين إذا كان مصدره مجهولًا؟"
  },
  {
    "question_id": 482,
    "category": "الأطعمة والذبائح والصيد",
    "question": "ما حكم استعمال الخل الناتج من تحول الخمر؟"
  },
  {
    "question_id": 483,
    "category": "الأطعمة والذبائح والصيد",
    "question": "هل يجوز أكل الأطعمة التي تحتوي على نسبة ضئيلة من الكحول؟"
  },
  {
    "question_id": 484,
    "category": "الأطعمة والذبائح والصيد",
    "question": "ما حكم أكل لحم الحيوان الذي مات بالصعق قبل الذبح؟"
  },
  {
    "question_id": 485,
    "category": "الأطعمة والذبائح والصيد",
    "question": "ما شروط الذكاة الشرعية للدواجن؟"
  },
  {
    "question_id": 486,
    "category": "الأطعمة والذبائح والصيد",
    "question": "هل يشترط أن يسمع الذابح التسمية بنفسه؟"
  },
  {
    "question_id": 487,
    "category": "الأطعمة والذبائح والصيد",
    "question": "ما حكم من نسي التسمية عند الذبح؟"
  },
  {
    "question_id": 488,
    "category": "الأطعمة والذبائح والصيد",
    "question": "هل يجوز ذبح الحيوان أمام حيوان آخر؟"
  },
  {
    "question_id": 489,
    "category": "الأطعمة والذبائح والصيد",
    "question": "ما حكم تعذيب الحيوان قبل ذبحه؟"
  },
  {
    "question_id": 490,
    "category": "الأطعمة والذبائح والصيد",
    "question": "هل يجوز صيد الطيور ببندقية الهواء؟"
  },
  {
    "question_id": 491,
    "category": "الأطعمة والذبائح والصيد",
    "question": "ما حكم أكل الصيد الذي سقط في الماء بعد إصابته؟"
  },
  {
    "question_id": 492,
    "category": "الأطعمة والذبائح والصيد",
    "question": "هل يجوز الصيد لمجرد التسلية دون حاجة؟"
  },
  {
    "question_id": 493,
    "category": "الأطعمة والذبائح والصيد",
    "question": "ما حكم بيع الحيوانات المحنطة؟"
  },
  {
    "question_id": 494,
    "category": "الأطعمة والذبائح والصيد",
    "question": "هل يجوز اقتناء حيوان مفترس في المنزل؟"
  },
  {
    "question_id": 495,
    "category": "الأطعمة والذبائح والصيد",
    "question": "ما حكم إطعام الحيوانات الأليفة طعامًا نجسًا؟"
  },
  {
    "question_id": 496,
    "category": "الأطعمة والذبائح والصيد",
    "question": "هل يجوز قتل الحشرات في البيت بالمبيدات؟"
  },
  {
    "question_id": 497,
    "category": "الأطعمة والذبائح والصيد",
    "question": "ما حكم قتل النمل إذا كان يؤذي أهل المنزل؟"
  },
  {
    "question_id": 498,
    "category": "الأطعمة والذبائح والصيد",
    "question": "هل يجوز تربية الكلاب لحراسة المزرعة؟"
  },
  {
    "question_id": 499,
    "category": "الأطعمة والذبائح والصيد",
    "question": "ما حكم بيع طعام مباح لمن يعلم أنه سيستخدمه في معصية؟"
  },
  {
    "question_id": 500,
    "category": "الأطعمة والذبائح والصيد",
    "question": "هل يجوز تناول طعام صنعه شخص يعتقد أن عليه بركة خاصة؟"
  },
  {
    "question_id": 501,
    "category": "اللباس والزينة والآداب",
    "question": "ما حكم لبس الرجل خاتم الذهب؟"
  },
  {
    "question_id": 502,
    "category": "اللباس والزينة والآداب",
    "question": "هل يجوز للرجل لبس ساعة مطلية بالذهب؟"
  },
  {
    "question_id": 503,
    "category": "اللباس والزينة والآداب",
    "question": "هل يجوز الصلاة بثوب عليه شعار ديني لغير المسلمين؟"
  },
  {
    "question_id": 504,
    "category": "اللباس والزينة والآداب",
    "question": "ما حكم لبس الثياب التي فيها تشبه بالجنس الآخر؟"
  },
  {
    "question_id": 505,
    "category": "اللباس والزينة والآداب",
    "question": "ما ضابط لباس الشهرة؟"
  },
  {
    "question_id": 506,
    "category": "اللباس والزينة والآداب",
    "question": "هل يجوز للمرأة لبس ثوب زفاف أبيض إذا كان عرفًا شائعًا؟"
  },
  {
    "question_id": 507,
    "category": "اللباس والزينة والآداب",
    "question": "هل يجوز للمرأة إزالة شعر الوجه؟"
  },
  {
    "question_id": 508,
    "category": "اللباس والزينة والآداب",
    "question": "ما حكم وصل الشعر بشعر صناعي؟"
  },
  {
    "question_id": 509,
    "category": "اللباس والزينة والآداب",
    "question": "هل يجوز صبغ الشعر باللون الأسود؟"
  },
  {
    "question_id": 510,
    "category": "اللباس والزينة والآداب",
    "question": "ما حكم الوشم المؤقت الذي يزول بعد أيام؟"
  },
  {
    "question_id": 511,
    "category": "اللباس والزينة والآداب",
    "question": "هل يجوز استخدام مستحضرات التجميل التي تمنع وصول الماء للبشرة؟"
  },
  {
    "question_id": 512,
    "category": "اللباس والزينة والآداب",
    "question": "ما حكم تركيب الأظافر الصناعية؟"
  },
  {
    "question_id": 513,
    "category": "اللباس والزينة والآداب",
    "question": "هل يجوز للمرأة قص شعرها قصًا قصيرًا؟"
  },
  {
    "question_id": 514,
    "category": "اللباس والزينة والآداب",
    "question": "ما حكم إزالة شعر الجسم بالليزر؟"
  },
  {
    "question_id": 515,
    "category": "اللباس والزينة والآداب",
    "question": "هل يجوز للرجل حلق بعض شعر الرأس وترك بعضه؟"
  },
  {
    "question_id": 516,
    "category": "اللباس والزينة والآداب",
    "question": "ما حكم لبس الثوب أسفل الكعبين؟"
  },
  {
    "question_id": 517,
    "category": "اللباس والزينة والآداب",
    "question": "هل يجوز للرجل لبس الحرير الصناعي؟"
  },
  {
    "question_id": 518,
    "category": "اللباس والزينة والآداب",
    "question": "ما حكم استعمال العطر الذي يحتوي على كحول؟"
  },
  {
    "question_id": 519,
    "category": "اللباس والزينة والآداب",
    "question": "هل يجوز وضع العطر للمرأة عند خروجها إلى مكان عام؟"
  },
  {
    "question_id": 520,
    "category": "اللباس والزينة والآداب",
    "question": "ما حكم التصوير للذكرى في المناسبات العائلية؟"
  },
  {
    "question_id": 521,
    "category": "اللباس والزينة والآداب",
    "question": "هل يجوز تعليق صور الأقارب في غرف المنزل؟"
  },
  {
    "question_id": 522,
    "category": "اللباس والزينة والآداب",
    "question": "ما حكم تقليد لباس شخصية مشهورة لغير قصد التشبه الديني؟"
  },
  {
    "question_id": 523,
    "category": "اللباس والزينة والآداب",
    "question": "هل يجوز استعمال أدوات التجميل المشتركة بين عدة أشخاص؟"
  },
  {
    "question_id": 524,
    "category": "الطب والتداوي والمرض",
    "question": "ما حكم التداوي بدواء يحتوي على مادة محرمة عند الضرورة؟"
  },
  {
    "question_id": 525,
    "category": "الطب والتداوي والمرض",
    "question": "ما ضابط الضرورة التي تبيح استعمال المحرم في العلاج؟"
  },
  {
    "question_id": 526,
    "category": "الطب والتداوي والمرض",
    "question": "هل يجوز ترك العلاج اعتمادًا على التوكل على الله؟"
  },
  {
    "question_id": 527,
    "category": "الطب والتداوي والمرض",
    "question": "ما حكم طلب الرقية من عدة أشخاص في الوقت نفسه؟"
  },
  {
    "question_id": 528,
    "category": "الطب والتداوي والمرض",
    "question": "هل يجوز الجمع بين العلاج الطبي والرقية الشرعية؟"
  },
  {
    "question_id": 529,
    "category": "الطب والتداوي والمرض",
    "question": "ما حكم التداوي بالحجامة في يوم معين اعتقادًا بفضله؟"
  },
  {
    "question_id": 530,
    "category": "الطب والتداوي والمرض",
    "question": "هل يجوز التبرع بالدم لإنقاذ مريض غير مسلم؟"
  },
  {
    "question_id": 531,
    "category": "الطب والتداوي والمرض",
    "question": "ما حكم نقل عضو من متوفى إلى مريض محتاج؟"
  },
  {
    "question_id": 532,
    "category": "الطب والتداوي والمرض",
    "question": "هل يجوز التبرع بالكلى في حياة المتبرع؟"
  },
  {
    "question_id": 533,
    "category": "الطب والتداوي والمرض",
    "question": "ما حكم التبرع بالجسم للبحث العلمي بعد الوفاة؟"
  },
  {
    "question_id": 534,
    "category": "الطب والتداوي والمرض",
    "question": "هل يجوز للطبيب كشف موضع من جسد المريض بقدر الحاجة؟"
  },
  {
    "question_id": 535,
    "category": "الطب والتداوي والمرض",
    "question": "ما حكم علاج الطبيب الرجل لامرأة عند وجود طبيبة؟"
  },
  {
    "question_id": 536,
    "category": "الطب والتداوي والمرض",
    "question": "هل يجوز للمريض الإفطار إذا نصحه الطبيب بذلك؟"
  },
  {
    "question_id": 537,
    "category": "الطب والتداوي والمرض",
    "question": "ما حكم الصيام مع تناول أدوية متعددة خلال النهار؟"
  },
  {
    "question_id": 538,
    "category": "الطب والتداوي والمرض",
    "question": "هل يجوز استعمال دواء تجريبي لم تثبت نتائجه بعد؟"
  },
  {
    "question_id": 539,
    "category": "الطب والتداوي والمرض",
    "question": "ما حكم إخفاء الطبيب خطأ طبيًا وقع فيه؟"
  },
  {
    "question_id": 540,
    "category": "الطب والتداوي والمرض",
    "question": "هل يجوز للمريض رفض إجراء طبي ضروري؟"
  },
  {
    "question_id": 541,
    "category": "الطب والتداوي والمرض",
    "question": "ما حكم إجراء عملية تجميل لإزالة تشوه خلقي؟"
  },
  {
    "question_id": 542,
    "category": "الطب والتداوي والمرض",
    "question": "هل يجوز إجراء عملية لتغيير مظهر طبيعي لمجرد عدم الرضا عنه؟"
  },
  {
    "question_id": 543,
    "category": "الطب والتداوي والمرض",
    "question": "ما حكم الإجهاض إذا ثبتت إصابة الجنين بمرض شديد؟"
  },
  {
    "question_id": 544,
    "category": "الطب والتداوي والمرض",
    "question": "هل يجوز تحديد جنس الجنين باستخدام التقنيات الطبية؟"
  },
  {
    "question_id": 545,
    "category": "الطب والتداوي والمرض",
    "question": "ما حكم التلقيح الصناعي إذا كانت البويضة والحيوان المنوي من الزوجين؟"
  },
  {
    "question_id": 546,
    "category": "الطب والتداوي والمرض",
    "question": "هل يجوز تجميد البويضات أو الحيوانات المنوية لأسباب علاجية؟"
  },
  {
    "question_id": 547,
    "category": "الطب والتداوي والمرض",
    "question": "ما حكم تأجير الأرحام؟"
  },
  {
    "question_id": 548,
    "category": "الطب والتداوي والمرض",
    "question": "هل يجوز للطبيب أخذ هدية من شركة أدوية مقابل وصف منتجاتها؟"
  },
  {
    "question_id": 549,
    "category": "السفر والإقامة والمواطنة",
    "question": "ما حكم السفر إلى بلد غير مسلم للدراسة؟"
  },
  {
    "question_id": 550,
    "category": "السفر والإقامة والمواطنة",
    "question": "ما الضوابط الشرعية للإقامة الطويلة في بلد غير مسلم؟"
  },
  {
    "question_id": 551,
    "category": "السفر والإقامة والمواطنة",
    "question": "هل يجوز السفر إلى بلد تنتشر فيه المعاصي للسياحة؟"
  },
  {
    "question_id": 552,
    "category": "السفر والإقامة والمواطنة",
    "question": "ما حكم السفر لطلب الرزق إذا كان العمل في بيئة مختلطة؟"
  },
  {
    "question_id": 553,
    "category": "السفر والإقامة والمواطنة",
    "question": "هل يجوز للمسلم الحصول على جنسية دولة غير مسلمة؟"
  },
  {
    "question_id": 554,
    "category": "السفر والإقامة والمواطنة",
    "question": "ما حكم طلب اللجوء إلى بلد غير مسلم عند الخوف على النفس؟"
  },
  {
    "question_id": 555,
    "category": "السفر والإقامة والمواطنة",
    "question": "هل يجوز العمل في جيش دولة غير مسلمة إذا كانت الوظيفة مدنية؟"
  },
  {
    "question_id": 556,
    "category": "السفر والإقامة والمواطنة",
    "question": "ما حكم أخذ إعانة حكومية لا يستحقها الشخص وفق الشروط؟"
  },
  {
    "question_id": 557,
    "category": "السفر والإقامة والمواطنة",
    "question": "هل يجوز مخالفة قوانين المرور إذا لم يوجد ضرر ظاهر؟"
  },
  {
    "question_id": 558,
    "category": "السفر والإقامة والمواطنة",
    "question": "ما حكم تزوير عنوان السكن للحصول على خدمة حكومية؟"
  },
  {
    "question_id": 559,
    "category": "السفر والإقامة والمواطنة",
    "question": "هل يجوز استعمال بطاقة نقل عامة لشخص آخر؟"
  },
  {
    "question_id": 560,
    "category": "السفر والإقامة والمواطنة",
    "question": "ما حكم تجاوز مدة التأشيرة بسبب تعذر العودة؟"
  },
  {
    "question_id": 561,
    "category": "السفر والإقامة والمواطنة",
    "question": "هل يجوز السفر بقصد زيارة مسجد معين مع اعتقاد فضل خاص له؟"
  },
  {
    "question_id": 562,
    "category": "السفر والإقامة والمواطنة",
    "question": "ما حكم السفر لحضور مؤتمر علمي يتضمن محاضرات محرمة؟"
  },
  {
    "question_id": 563,
    "category": "السفر والإقامة والمواطنة",
    "question": "هل يجوز للمسافر جمع الصلوات بسبب مشقة المطار؟"
  },
  {
    "question_id": 564,
    "category": "السفر والإقامة والمواطنة",
    "question": "ما حكم الصلاة في الطائرة مع القدرة على النزول قبل خروج الوقت؟"
  },
  {
    "question_id": 565,
    "category": "السفر والإقامة والمواطنة",
    "question": "كيف يصلي المسافر إذا لم يعرف اتجاه القبلة في الطائرة؟"
  },
  {
    "question_id": 566,
    "category": "السفر والإقامة والمواطنة",
    "question": "ما حكم الصيام في رحلة طويلة تعبر عدة مناطق زمنية؟"
  },
  {
    "question_id": 567,
    "category": "السفر والإقامة والمواطنة",
    "question": "هل يجوز الإفطار بسبب مشقة السفر القصير؟"
  },
  {
    "question_id": 568,
    "category": "السفر والإقامة والمواطنة",
    "question": "ما حكم الصلاة على كرسي في القطار عند عدم وجود مكان للوقوف؟"
  },
  {
    "question_id": 569,
    "category": "السفر والإقامة والمواطنة",
    "question": "هل يجوز للمسلم ترك صلاة الجماعة بسبب دوام السفر؟"
  },
  {
    "question_id": 570,
    "category": "السفر والإقامة والمواطنة",
    "question": "ما حكم استخدام مرافق عامة مخصصة لفئة أخرى عند الضرورة؟"
  },
  {
    "question_id": 571,
    "category": "السفر والإقامة والمواطنة",
    "question": "هل يجوز قبول هدية من موظف الجمارك لتسهيل إجراء قانوني؟"
  },
  {
    "question_id": 572,
    "category": "السفر والإقامة والمواطنة",
    "question": "ما حكم دفع مال لتسريع معاملة حكومية مستحقة؟"
  },
  {
    "question_id": 573,
    "category": "السفر والإقامة والمواطنة",
    "question": "هل يجوز أخذ تعويض عن تأخر رحلة جوية وفق شروط الشركة؟"
  },
  {
    "question_id": 574,
    "category": "المسجد والجمعة والعيد",
    "question": "ما حكم الكلام أثناء خطبة الجمعة؟"
  },
  {
    "question_id": 575,
    "category": "المسجد والجمعة والعيد",
    "question": "هل يجوز لمن دخل المسجد والإمام يخطب أن يصلي ركعتين؟"
  },
  {
    "question_id": 576,
    "category": "المسجد والجمعة والعيد",
    "question": "ما حكم تخطي رقاب الناس يوم الجمعة؟"
  },
  {
    "question_id": 577,
    "category": "المسجد والجمعة والعيد",
    "question": "هل يشرع الاغتسال للجمعة لمن لا يخرج إلى المسجد؟"
  },
  {
    "question_id": 578,
    "category": "المسجد والجمعة والعيد",
    "question": "ما حكم البيع والشراء بعد النداء الثاني للجمعة؟"
  },
  {
    "question_id": 579,
    "category": "المسجد والجمعة والعيد",
    "question": "هل تصح الجمعة إذا وافقت يوم عيد؟"
  },
  {
    "question_id": 580,
    "category": "المسجد والجمعة والعيد",
    "question": "ما حكم تعدد صلاة الجمعة في حي صغير؟"
  },
  {
    "question_id": 581,
    "category": "المسجد والجمعة والعيد",
    "question": "هل يجوز تأخير خطبة الجمعة عن وقتها المعتاد لحاجة الناس؟"
  },
  {
    "question_id": 582,
    "category": "المسجد والجمعة والعيد",
    "question": "ما حكم ترجمة خطبة الجمعة إلى لغة يفهمها المصلون؟"
  },
  {
    "question_id": 583,
    "category": "المسجد والجمعة والعيد",
    "question": "هل يجوز استخدام مكبرات الصوت الخارجية للخطبة إذا آذت الجيران؟"
  },
  {
    "question_id": 584,
    "category": "المسجد والجمعة والعيد",
    "question": "ما حكم حجز مكان في الصف الأول بوضع سجادة ثم الخروج؟"
  },
  {
    "question_id": 585,
    "category": "المسجد والجمعة والعيد",
    "question": "هل يجوز المرور أمام المصلين في المسجد إذا ضاق المكان؟"
  },
  {
    "question_id": 586,
    "category": "المسجد والجمعة والعيد",
    "question": "ما حكم رفع الصوت بالقراءة بما يشوش على المصلين؟"
  },
  {
    "question_id": 587,
    "category": "المسجد والجمعة والعيد",
    "question": "هل يجوز تخصيص درس بعد كل صلاة في المسجد؟"
  },
  {
    "question_id": 588,
    "category": "المسجد والجمعة والعيد",
    "question": "ما حكم جمع التبرعات في المسجد أثناء خطبة الجمعة؟"
  },
  {
    "question_id": 589,
    "category": "المسجد والجمعة والعيد",
    "question": "هل يجوز النوم في المسجد لمن ليس معتكفًا؟"
  },
  {
    "question_id": 590,
    "category": "المسجد والجمعة والعيد",
    "question": "ما حكم الأكل في المسجد إذا خشي تلويثه؟"
  },
  {
    "question_id": 591,
    "category": "المسجد والجمعة والعيد",
    "question": "هل يجوز إدخال الأطفال إلى المسجد إذا كانوا يشوشون؟"
  },
  {
    "question_id": 592,
    "category": "المسجد والجمعة والعيد",
    "question": "ما حكم اصطحاب طفل صغير إلى صلاة الجماعة؟"
  },
  {
    "question_id": 593,
    "category": "المسجد والجمعة والعيد",
    "question": "هل يجوز للمرأة حضور صلاة العيد في المصلى؟"
  },
  {
    "question_id": 594,
    "category": "المسجد والجمعة والعيد",
    "question": "ما حكم التكبير الجماعي في العيد بصوت واحد؟"
  },
  {
    "question_id": 595,
    "category": "المسجد والجمعة والعيد",
    "question": "هل ثبتت صلاة ركعتين قبل صلاة العيد في المصلى؟"
  },
  {
    "question_id": 596,
    "category": "المسجد والجمعة والعيد",
    "question": "ما حكم تهنئة الناس بالعيد بعبارات معتادة؟"
  },
  {
    "question_id": 597,
    "category": "المسجد والجمعة والعيد",
    "question": "هل يجوز زيارة الأقارب في يوم العيد مع اعتقاد أنها عادة لا عبادة؟"
  },
  {
    "question_id": 598,
    "category": "المسجد والجمعة والعيد",
    "question": "ما حكم تخصيص يوم العيد بزيارة المقابر؟"
  },
  {
    "question_id": 599,
    "category": "الآداب والعلاقات الاجتماعية",
    "question": "ما حكم إفشاء السر الذي اؤتمن عليه الإنسان؟"
  },
  {
    "question_id": 600,
    "category": "الآداب والعلاقات الاجتماعية",
    "question": "هل يجوز نقل الكلام بين شخصين بقصد الإصلاح؟"
  },
  {
    "question_id": 601,
    "category": "الآداب والعلاقات الاجتماعية",
    "question": "ما حكم الغيبة إذا كان الكلام صحيحًا؟"
  },
  {
    "question_id": 602,
    "category": "الآداب والعلاقات الاجتماعية",
    "question": "متى تجوز الشكوى من شخص اغتابني؟"
  },
  {
    "question_id": 603,
    "category": "الآداب والعلاقات الاجتماعية",
    "question": "ما حكم التجسس على هاتف الزوج أو الزوجة؟"
  },
  {
    "question_id": 604,
    "category": "الآداب والعلاقات الاجتماعية",
    "question": "هل يجوز قراءة رسائل شخص آخر دون إذنه؟"
  },
  {
    "question_id": 605,
    "category": "الآداب والعلاقات الاجتماعية",
    "question": "ما حكم المزاح بالكذب لإضحاك الآخرين؟"
  },
  {
    "question_id": 606,
    "category": "الآداب والعلاقات الاجتماعية",
    "question": "هل يجوز إطلاق الألقاب التي يكرهها أصحابها؟"
  },
  {
    "question_id": 607,
    "category": "الآداب والعلاقات الاجتماعية",
    "question": "ما حكم السخرية من لهجة شخص أو طريقة كلامه؟"
  },
  {
    "question_id": 608,
    "category": "الآداب والعلاقات الاجتماعية",
    "question": "هل يجوز مقاطعة شخص أثناء حديثه لتصحيح خطئه؟"
  },
  {
    "question_id": 609,
    "category": "الآداب والعلاقات الاجتماعية",
    "question": "ما حكم رد الهدية بعد قبولها؟"
  },
  {
    "question_id": 610,
    "category": "الآداب والعلاقات الاجتماعية",
    "question": "هل يجب رد الهدية إذا طلبها صاحبها لاحقًا؟"
  },
  {
    "question_id": 611,
    "category": "الآداب والعلاقات الاجتماعية",
    "question": "ما حكم قبول هدية من شخص يريد استمالة صاحب سلطة؟"
  },
  {
    "question_id": 612,
    "category": "الآداب والعلاقات الاجتماعية",
    "question": "هل يجوز الاعتذار عن دعوة وليمة بلا عذر؟"
  },
  {
    "question_id": 613,
    "category": "الآداب والعلاقات الاجتماعية",
    "question": "ما حكم حضور مناسبة عائلية فيها منكر مع عدم القدرة على تغييره؟"
  },
  {
    "question_id": 614,
    "category": "الآداب والعلاقات الاجتماعية",
    "question": "هل يجوز الجلوس في مجلس يغلب عليه الغيبة إذا لم يشارك فيها؟"
  },
  {
    "question_id": 615,
    "category": "الآداب والعلاقات الاجتماعية",
    "question": "ما حكم نشر خلاف عائلي على وسائل التواصل؟"
  },
  {
    "question_id": 616,
    "category": "الآداب والعلاقات الاجتماعية",
    "question": "هل يجوز الاستعانة بوسيط للإصلاح بين زوجين؟"
  },
  {
    "question_id": 617,
    "category": "الآداب والعلاقات الاجتماعية",
    "question": "ما حكم رد الإساءة بمثلها دون تجاوز؟"
  },
  {
    "question_id": 618,
    "category": "الآداب والعلاقات الاجتماعية",
    "question": "هل يجوز الامتناع عن مصافحة قريب لأسباب دينية؟"
  },
  {
    "question_id": 619,
    "category": "الآداب والعلاقات الاجتماعية",
    "question": "ما حكم مصافحة المرأة الأجنبية عند الإحراج الاجتماعي؟"
  },
  {
    "question_id": 620,
    "category": "الآداب والعلاقات الاجتماعية",
    "question": "هل يجوز إعطاء جار غير مسلم من طعام العيد؟"
  },
  {
    "question_id": 621,
    "category": "الآداب والعلاقات الاجتماعية",
    "question": "ما حكم قبول دعوة طعام من جار غير مسلم؟"
  },
  {
    "question_id": 622,
    "category": "الآداب والعلاقات الاجتماعية",
    "question": "هل يجب إجابة دعوة الزواج إذا كان فيها منكر ظاهر؟"
  },
  {
    "question_id": 623,
    "category": "الآداب والعلاقات الاجتماعية",
    "question": "ما حكم الاعتذار عن خدمة طلبها قريب إذا كان فيها مشقة شديدة؟"
  },
  {
    "question_id": 624,
    "category": "العمل والتوظيف والأمانة المهنية",
    "question": "ما حكم كتابة سيرة ذاتية تتضمن مهارات لم يتقنها المتقدم؟"
  },
  {
    "question_id": 625,
    "category": "العمل والتوظيف والأمانة المهنية",
    "question": "هل يجوز المبالغة في وصف الخبرة المهنية للحصول على وظيفة؟"
  },
  {
    "question_id": 626,
    "category": "العمل والتوظيف والأمانة المهنية",
    "question": "ما حكم إخفاء سبب ترك الوظيفة السابقة عند التقديم لعمل جديد؟"
  },
  {
    "question_id": 627,
    "category": "العمل والتوظيف والأمانة المهنية",
    "question": "هل يجوز استخدام برامج مقرصنة في العمل؟"
  },
  {
    "question_id": 628,
    "category": "العمل والتوظيف والأمانة المهنية",
    "question": "ما حكم نسخ ملفات الشركة إلى جهاز شخصي دون إذن؟"
  },
  {
    "question_id": 629,
    "category": "العمل والتوظيف والأمانة المهنية",
    "question": "هل يجوز استخدام بيانات العملاء لأغراض شخصية؟"
  },
  {
    "question_id": 630,
    "category": "العمل والتوظيف والأمانة المهنية",
    "question": "ما حكم العمل في شركة تنتج محتوى مباحًا ومحرمًا معًا؟"
  },
  {
    "question_id": 631,
    "category": "العمل والتوظيف والأمانة المهنية",
    "question": "هل يجوز قبول وظيفة تتطلب الكذب على العملاء؟"
  },
  {
    "question_id": 632,
    "category": "العمل والتوظيف والأمانة المهنية",
    "question": "ما حكم الموظف الذي يتقاضى راتبًا مع قلة العمل بسبب الإدارة؟"
  },
  {
    "question_id": 633,
    "category": "العمل والتوظيف والأمانة المهنية",
    "question": "هل يجوز العمل عن بعد من بلد آخر دون إبلاغ صاحب العمل؟"
  },
  {
    "question_id": 634,
    "category": "العمل والتوظيف والأمانة المهنية",
    "question": "ما حكم تسجيل الحضور في العمل ثم الخروج لقضاء حاجة شخصية؟"
  },
  {
    "question_id": 635,
    "category": "العمل والتوظيف والأمانة المهنية",
    "question": "هل يجوز استخدام الذكاء الاصطناعي لإنجاز عمل وظيفي إذا لم يمنعه العقد؟"
  },
  {
    "question_id": 636,
    "category": "العمل والتوظيف والأمانة المهنية",
    "question": "ما حكم تسليم عمل أنجزه زميل على أنه من إنجازي؟"
  },
  {
    "question_id": 637,
    "category": "العمل والتوظيف والأمانة المهنية",
    "question": "هل يجوز أخذ عمولة سرية من مورد مقابل ترسية عقد؟"
  },
  {
    "question_id": 638,
    "category": "العمل والتوظيف والأمانة المهنية",
    "question": "ما حكم التنافس على وظيفة باستخدام معلومات داخلية مسربة؟"
  },
  {
    "question_id": 639,
    "category": "العمل والتوظيف والأمانة المهنية",
    "question": "هل يجوز للموظف قبول مكافأة من عميل بعد انتهاء المشروع؟"
  },
  {
    "question_id": 640,
    "category": "العمل والتوظيف والأمانة المهنية",
    "question": "ما حكم الامتناع عن العمل الإضافي إذا لم يكن منصوصًا عليه؟"
  },
  {
    "question_id": 641,
    "category": "العمل والتوظيف والأمانة المهنية",
    "question": "هل يجوز ترك العمل فجأة إذا لم يدفع صاحب العمل الراتب؟"
  },
  {
    "question_id": 642,
    "category": "العمل والتوظيف والأمانة المهنية",
    "question": "ما حكم أخذ إجازة مرضية دون وجود مرض حقيقي؟"
  },
  {
    "question_id": 643,
    "category": "العمل والتوظيف والأمانة المهنية",
    "question": "هل يجوز إخفاء خطأ في منتج عن الإدارة خوفًا من العقوبة؟"
  },
  {
    "question_id": 644,
    "category": "العمل والتوظيف والأمانة المهنية",
    "question": "ما حكم مشاركة كلمة مرور حساب العمل مع زميل؟"
  },
  {
    "question_id": 645,
    "category": "العمل والتوظيف والأمانة المهنية",
    "question": "هل يجوز العمل في منصة تعرض إعلانات محرمة أحيانًا؟"
  },
  {
    "question_id": 646,
    "category": "العمل والتوظيف والأمانة المهنية",
    "question": "ما حكم تصميم موقع لعميل يبيع منتجات مباحة ويعرض صورًا محرمة؟"
  },
  {
    "question_id": 647,
    "category": "العمل والتوظيف والأمانة المهنية",
    "question": "هل يجوز استخدام وقت العمل لتعلم مهارة تخدم الوظيفة مستقبلًا؟"
  },
  {
    "question_id": 648,
    "category": "العمل والتوظيف والأمانة المهنية",
    "question": "ما حكم أخذ مواد مكتبية بسيطة من مقر العمل للاستخدام الشخصي؟"
  },
  {
    "question_id": 649,
    "category": "التعليم والامتحانات والبحث",
    "question": "ما حكم الغش في الامتحان إذا كان الطالب يعرف المادة لكنه يخشى الرسوب؟"
  },
  {
    "question_id": 650,
    "category": "التعليم والامتحانات والبحث",
    "question": "هل يجوز مساعدة زميل في امتحان منزلي إذا سمحت التعليمات بالتعاون؟"
  },
  {
    "question_id": 651,
    "category": "التعليم والامتحانات والبحث",
    "question": "ما حكم شراء بحث جامعي جاهز وتقديمه باسم الطالب؟"
  },
  {
    "question_id": 652,
    "category": "التعليم والامتحانات والبحث",
    "question": "هل يجوز استخدام أدوات الذكاء الاصطناعي في البحث إذا منع الأستاذ ذلك؟"
  },
  {
    "question_id": 653,
    "category": "التعليم والامتحانات والبحث",
    "question": "ما حكم نقل فقرة من كتاب دون توثيق المصدر؟"
  },
  {
    "question_id": 654,
    "category": "التعليم والامتحانات والبحث",
    "question": "هل يجوز تسجيل المحاضرة دون إذن المدرس؟"
  },
  {
    "question_id": 655,
    "category": "التعليم والامتحانات والبحث",
    "question": "ما حكم بيع ملخصات دراسية أعدها الطالب بنفسه؟"
  },
  {
    "question_id": 656,
    "category": "التعليم والامتحانات والبحث",
    "question": "هل يجوز تداول أسئلة امتحان مسربة بين الطلاب؟"
  },
  {
    "question_id": 657,
    "category": "التعليم والامتحانات والبحث",
    "question": "ما حكم دفع مال لشخص ليكتب مشروع التخرج كاملًا؟"
  },
  {
    "question_id": 658,
    "category": "التعليم والامتحانات والبحث",
    "question": "هل يجوز إعادة استخدام بحث سابق في مقرر جديد دون تصريح؟"
  },
  {
    "question_id": 659,
    "category": "التعليم والامتحانات والبحث",
    "question": "ما حكم تعديل نتائج تجربة علمية لتوافق الفرضية؟"
  },
  {
    "question_id": 660,
    "category": "التعليم والامتحانات والبحث",
    "question": "هل يجوز حذف بيانات بحثية لا تدعم النتيجة المقصودة؟"
  },
  {
    "question_id": 661,
    "category": "التعليم والامتحانات والبحث",
    "question": "ما حكم نشر بحث مشترك دون ذكر أحد المشاركين؟"
  },
  {
    "question_id": 662,
    "category": "التعليم والامتحانات والبحث",
    "question": "هل يجوز إدراج اسم أستاذ في بحث لم يشارك فيه؟"
  },
  {
    "question_id": 663,
    "category": "التعليم والامتحانات والبحث",
    "question": "ما حكم قبول منحة دراسية مع عدم استيفاء أحد شروطها؟"
  },
  {
    "question_id": 664,
    "category": "التعليم والامتحانات والبحث",
    "question": "هل يجوز تقديم شهادة لغة غير حقيقية للحصول على قبول جامعي؟"
  },
  {
    "question_id": 665,
    "category": "التعليم والامتحانات والبحث",
    "question": "ما حكم التقديم على منحة مع العلم أن التمويل قد يستخدم في محرم؟"
  },
  {
    "question_id": 666,
    "category": "التعليم والامتحانات والبحث",
    "question": "هل يجوز للطالب الغياب عن المحاضرات مع حضور الامتحان فقط؟"
  },
  {
    "question_id": 667,
    "category": "التعليم والامتحانات والبحث",
    "question": "ما حكم تأخير تسليم مشروع جماعي بسبب تقصير عضو واحد؟"
  },
  {
    "question_id": 668,
    "category": "التعليم والامتحانات والبحث",
    "question": "هل يجوز للمدرس إعطاء درجات إضافية لطالب بسبب ظروفه الخاصة؟"
  },
  {
    "question_id": 669,
    "category": "التعليم والامتحانات والبحث",
    "question": "ما حكم تصحيح المعلم لأوراق أقاربه؟"
  },
  {
    "question_id": 670,
    "category": "التعليم والامتحانات والبحث",
    "question": "هل يجوز نشر حلول واجبات مدفوعة على الإنترنت؟"
  },
  {
    "question_id": 671,
    "category": "التعليم والامتحانات والبحث",
    "question": "ما حكم بيع حساب تعليمي حصل عليه الطالب مجانًا؟"
  },
  {
    "question_id": 672,
    "category": "التعليم والامتحانات والبحث",
    "question": "هل يجوز استخدام كتاب إلكتروني منسوخ إذا تعذر شراؤه؟"
  },
  {
    "question_id": 673,
    "category": "التعليم والامتحانات والبحث",
    "question": "ما حكم طلب إعادة تصحيح علامة يظن الطالب أنها غير عادلة؟"
  },
  {
    "question_id": 674,
    "category": "التقنية والإنترنت والخصوصية",
    "question": "ما حكم اختراق حساب شخص لاستعادة مال يدعي أنه أخذه؟"
  },
  {
    "question_id": 675,
    "category": "التقنية والإنترنت والخصوصية",
    "question": "هل يجوز اختبار أمان موقع دون إذن مالكه؟"
  },
  {
    "question_id": 676,
    "category": "التقنية والإنترنت والخصوصية",
    "question": "ما حكم نشر ثغرة أمنية قبل إبلاغ الجهة المسؤولة؟"
  },
  {
    "question_id": 677,
    "category": "التقنية والإنترنت والخصوصية",
    "question": "هل يجوز استخدام صورة شخص في حساب ساخر دون موافقته؟"
  },
  {
    "question_id": 678,
    "category": "التقنية والإنترنت والخصوصية",
    "question": "ما حكم إنشاء حساب باسم مستعار للتعبير عن الرأي؟"
  },
  {
    "question_id": 679,
    "category": "التقنية والإنترنت والخصوصية",
    "question": "هل يجوز انتحال شخصية شخص آخر للمزاح؟"
  },
  {
    "question_id": 680,
    "category": "التقنية والإنترنت والخصوصية",
    "question": "ما حكم شراء متابعين وهميين لحساب تجاري؟"
  },
  {
    "question_id": 681,
    "category": "التقنية والإنترنت والخصوصية",
    "question": "هل يجوز استخدام بوتات لزيادة التفاعل إذا خالف ذلك شروط المنصة؟"
  },
  {
    "question_id": 682,
    "category": "التقنية والإنترنت والخصوصية",
    "question": "ما حكم نشر مقطع لشخص في موقف محرج دون إذنه؟"
  },
  {
    "question_id": 683,
    "category": "التقنية والإنترنت والخصوصية",
    "question": "هل يجوز الاحتفاظ بصور خاصة أرسلها شخص ثم انتهت العلاقة؟"
  },
  {
    "question_id": 684,
    "category": "التقنية والإنترنت والخصوصية",
    "question": "ما حكم مشاركة موقع شخص الجغرافي دون علمه؟"
  },
  {
    "question_id": 685,
    "category": "التقنية والإنترنت والخصوصية",
    "question": "هل يجوز قراءة محادثة عامة في مجموعة ثم نشرها خارج المجموعة؟"
  },
  {
    "question_id": 686,
    "category": "التقنية والإنترنت والخصوصية",
    "question": "ما حكم بيع بيانات المستخدمين لشركات الإعلان؟"
  },
  {
    "question_id": 687,
    "category": "التقنية والإنترنت والخصوصية",
    "question": "هل يجوز استخدام برنامج يمنع الإعلانات على موقع يعتمد عليها؟"
  },
  {
    "question_id": 688,
    "category": "التقنية والإنترنت والخصوصية",
    "question": "ما حكم إنشاء محتوى آلي ينسب أقوالًا مختلقة إلى أشخاص؟"
  },
  {
    "question_id": 689,
    "category": "التقنية والإنترنت والخصوصية",
    "question": "هل يجوز نشر إجابة آلية شرعية دون مراجعتها؟"
  },
  {
    "question_id": 690,
    "category": "التقنية والإنترنت والخصوصية",
    "question": "ما حكم الاعتماد على روبوت محادثة في معرفة حكم مسألة شرعية؟"
  },
  {
    "question_id": 691,
    "category": "التقنية والإنترنت والخصوصية",
    "question": "هل يجوز استخدام صوت شخص مولد بالذكاء الاصطناعي دون إذنه؟"
  },
  {
    "question_id": 692,
    "category": "التقنية والإنترنت والخصوصية",
    "question": "ما حكم إنشاء صورة واقعية مزيفة لشخص بغرض المزاح؟"
  },
  {
    "question_id": 693,
    "category": "التقنية والإنترنت والخصوصية",
    "question": "هل يجوز نشر مراجعة كاذبة لمنتج مقابل هدية؟"
  },
  {
    "question_id": 694,
    "category": "التقنية والإنترنت والخصوصية",
    "question": "ما حكم مشاركة كلمات المرور مع أفراد الأسرة؟"
  },
  {
    "question_id": 695,
    "category": "التقنية والإنترنت والخصوصية",
    "question": "هل يجوز تجاوز القيود الجغرافية لخدمة رقمية مدفوعة؟"
  },
  {
    "question_id": 696,
    "category": "التقنية والإنترنت والخصوصية",
    "question": "ما حكم تحميل كتاب إلكتروني من مصدر غير رسمي مع توفره للبيع؟"
  },
  {
    "question_id": 697,
    "category": "التقنية والإنترنت والخصوصية",
    "question": "هل يجوز استخدام بيانات شخصية عامة لأغراض بحثية؟"
  },
  {
    "question_id": 698,
    "category": "التقنية والإنترنت والخصوصية",
    "question": "ما حكم نشر محادثة خاصة لإثبات أن شخصًا كذب؟"
  },
  {
    "question_id": 699,
    "category": "العبادات القلبية والتوبة",
    "question": "ما شروط التوبة النصوح من ذنب يتعلق بحقوق الناس؟"
  },
  {
    "question_id": 700,
    "category": "العبادات القلبية والتوبة",
    "question": "هل تصح التوبة إذا عاد الإنسان إلى الذنب بعد مدة؟"
  },
  {
    "question_id": 701,
    "category": "العبادات القلبية والتوبة",
    "question": "ما حكم تأخير التوبة بحجة أن الإنسان ما زال شابًا؟"
  },
  {
    "question_id": 702,
    "category": "العبادات القلبية والتوبة",
    "question": "كيف يتصرف من ندم على معصية لكنه لا يستطيع رد مظلمته كاملة؟"
  },
  {
    "question_id": 703,
    "category": "العبادات القلبية والتوبة",
    "question": "هل يجب إخبار الناس بالذنب القديم بعد التوبة منه؟"
  },
  {
    "question_id": 704,
    "category": "العبادات القلبية والتوبة",
    "question": "ما حكم المجاهرة بالمعصية بعد الوقوع فيها؟"
  },
  {
    "question_id": 705,
    "category": "العبادات القلبية والتوبة",
    "question": "هل يجوز للإنسان أن يذكر معصية سابقة للتنبيه والتحذير؟"
  },
  {
    "question_id": 706,
    "category": "العبادات القلبية والتوبة",
    "question": "ما الفرق بين الرياء والعجب؟"
  },
  {
    "question_id": 707,
    "category": "العبادات القلبية والتوبة",
    "question": "كيف يعالج المسلم خوفه من الرياء في العبادة؟"
  },
  {
    "question_id": 708,
    "category": "العبادات القلبية والتوبة",
    "question": "ما حكم ترك العمل الصالح خشية أن يراه الناس؟"
  },
  {
    "question_id": 709,
    "category": "العبادات القلبية والتوبة",
    "question": "هل يجوز إظهار الفرح بالطاعة أمام الأصدقاء تشجيعًا لهم؟"
  },
  {
    "question_id": 710,
    "category": "العبادات القلبية والتوبة",
    "question": "ما حكم مقارنة النفس بالآخرين في كثرة العبادة؟"
  },
  {
    "question_id": 711,
    "category": "العبادات القلبية والتوبة",
    "question": "كيف يتعامل المسلم مع الفتور بعد فترة من النشاط؟"
  },
  {
    "question_id": 712,
    "category": "العبادات القلبية والتوبة",
    "question": "ما حكم اليأس من رحمة الله بسبب تكرار الذنب؟"
  },
  {
    "question_id": 713,
    "category": "العبادات القلبية والتوبة",
    "question": "هل يؤاخذ الإنسان على خواطر سيئة لم يتكلم بها ولم يعمل؟"
  },
  {
    "question_id": 714,
    "category": "العبادات القلبية والتوبة",
    "question": "ما حكم الوسوسة في النية قبل العبادة؟"
  },
  {
    "question_id": 715,
    "category": "العبادات القلبية والتوبة",
    "question": "هل يلزم التلفظ بالنية عند الصيام أو الصلاة؟"
  },
  {
    "question_id": 716,
    "category": "العبادات القلبية والتوبة",
    "question": "ما حكم استحضار النية في كل جزء من العبادة؟"
  },
  {
    "question_id": 717,
    "category": "العبادات القلبية والتوبة",
    "question": "كيف يفرق المسلم بين حسن الظن بالله والاغترار بالرحمة؟"
  },
  {
    "question_id": 718,
    "category": "العبادات القلبية والتوبة",
    "question": "ما حكم الفرح بثناء الناس على العمل الصالح بعد الفراغ منه؟"
  },
  {
    "question_id": 719,
    "category": "العبادات القلبية والتوبة",
    "question": "هل يجوز أن يسأل المسلم الناس الدعاء له بالثبات؟"
  },
  {
    "question_id": 720,
    "category": "العبادات القلبية والتوبة",
    "question": "ما حكم الحزن على فوات نافلة اعتادها المسلم؟"
  },
  {
    "question_id": 721,
    "category": "العبادات القلبية والتوبة",
    "question": "هل يؤجر المرء على نية العمل الصالح إذا منعه عذر؟"
  },
  {
    "question_id": 722,
    "category": "العبادات القلبية والتوبة",
    "question": "ما حكم المداومة على عبادة قليلة مع ترك أعمال نافلة كثيرة؟"
  },
  {
    "question_id": 723,
    "category": "العبادات القلبية والتوبة",
    "question": "كيف يتعامل المسلم مع شعور القسوة وقلة التأثر بالمواعظ؟"
  },
  {
    "question_id": 724,
    "category": "المرأة والأسرة: مسائل إضافية",
    "question": "ما حكم تعليم المرأة للرجال عبر منصة إلكترونية مع وجود ضوابط؟"
  },
  {
    "question_id": 725,
    "category": "المرأة والأسرة: مسائل إضافية",
    "question": "هل يجوز للمرأة أن تتولى منصبًا إداريًا يشمل رجالًا ونساء؟"
  },
  {
    "question_id": 726,
    "category": "المرأة والأسرة: مسائل إضافية",
    "question": "ما حكم سفر المرأة مع مجموعة مأمونة دون محرم؟"
  },
  {
    "question_id": 727,
    "category": "المرأة والأسرة: مسائل إضافية",
    "question": "هل يجوز للمرأة أن ترفض الحمل مدة قصيرة لأسباب صحية؟"
  },
  {
    "question_id": 728,
    "category": "المرأة والأسرة: مسائل إضافية",
    "question": "ما حكم كشف المرأة شعرها أمام النساء غير المسلمات؟"
  },
  {
    "question_id": 729,
    "category": "المرأة والأسرة: مسائل إضافية",
    "question": "هل يجوز للمرأة أن تقص شعرها للتداوي من مرض في فروة الرأس؟"
  },
  {
    "question_id": 730,
    "category": "المرأة والأسرة: مسائل إضافية",
    "question": "ما حكم استعمال المرأة عطرًا في بيتها بحضور محارمها؟"
  },
  {
    "question_id": 731,
    "category": "المرأة والأسرة: مسائل إضافية",
    "question": "هل يجوز للمرأة أن تخرج للعمل إذا كان زوجها عاجزًا عن النفقة؟"
  },
  {
    "question_id": 732,
    "category": "المرأة والأسرة: مسائل إضافية",
    "question": "ما حكم اشتراط الزوجين عدم الانتقال من المدينة في عقد الزواج؟"
  },
  {
    "question_id": 733,
    "category": "المرأة والأسرة: مسائل إضافية",
    "question": "هل يجوز للزوج أن يمنع زوجته من زيارة صديقتها الصالحة؟"
  },
  {
    "question_id": 734,
    "category": "المرأة والأسرة: مسائل إضافية",
    "question": "ما حكم إساءة الزوج اللفظية المتكررة إلى زوجته؟"
  },
  {
    "question_id": 735,
    "category": "المرأة والأسرة: مسائل إضافية",
    "question": "كيف تتصرف الزوجة إذا امتنع الزوج عن أداء حقوقها دون سبب؟"
  },
  {
    "question_id": 736,
    "category": "المرأة والأسرة: مسائل إضافية",
    "question": "ما حكم طلب الزوج من زوجته التنازل عن مهرها بعد العقد؟"
  },
  {
    "question_id": 737,
    "category": "المرأة والأسرة: مسائل إضافية",
    "question": "هل يجوز تأخير دفع المهر المؤجل عند قدرة الزوج؟"
  },
  {
    "question_id": 738,
    "category": "المرأة والأسرة: مسائل إضافية",
    "question": "ما حكم رد الزوجة إلى بيت الزوجية بعد نزاع دون معالجة أسباب الخلاف؟"
  },
  {
    "question_id": 739,
    "category": "المرأة والأسرة: مسائل إضافية",
    "question": "هل يجوز للزوجين الاتفاق على توزيع أعمال المنزل بينهما؟"
  },
  {
    "question_id": 740,
    "category": "المرأة والأسرة: مسائل إضافية",
    "question": "ما حكم أخذ الأم من مال ابنها البالغ دون إذنه؟"
  },
  {
    "question_id": 741,
    "category": "المرأة والأسرة: مسائل إضافية",
    "question": "هل يجب على الابن الإنفاق على والديه إذا كان دخله محدودًا؟"
  },
  {
    "question_id": 742,
    "category": "المرأة والأسرة: مسائل إضافية",
    "question": "ما حكم منع الأبناء من صلة أقارب أمهم بعد الطلاق؟"
  },
  {
    "question_id": 743,
    "category": "المرأة والأسرة: مسائل إضافية",
    "question": "هل يجوز للأبوين التفريق في الهدايا بين الأبناء لاختلاف حاجاتهم؟"
  },
  {
    "question_id": 744,
    "category": "المرأة والأسرة: مسائل إضافية",
    "question": "ما حكم إعطاء طفل هدية مقابل حفظ القرآن؟"
  },
  {
    "question_id": 745,
    "category": "المرأة والأسرة: مسائل إضافية",
    "question": "هل يجوز تأديب الطفل بالضرب الخفيف إذا لم تنفع الوسائل الأخرى؟"
  },
  {
    "question_id": 746,
    "category": "المرأة والأسرة: مسائل إضافية",
    "question": "ما ضوابط حضانة الطفل إذا كان أحد الوالدين يسكن بعيدًا؟"
  },
  {
    "question_id": 747,
    "category": "المرأة والأسرة: مسائل إضافية",
    "question": "ما حكم نشر صور الأطفال على الإنترنت دون مراعاة خصوصيتهم؟"
  },
  {
    "question_id": 748,
    "category": "المرأة والأسرة: مسائل إضافية",
    "question": "هل يجوز للوالدين إلزام الابن البالغ بتخصص لا يرغب فيه؟"
  },
  {
    "question_id": 749,
    "category": "الحيض والاستحاضة والنفاس",
    "question": "ما حكم المرأة التي ترى صفرة بعد انقطاع دم الحيض؟"
  },
  {
    "question_id": 750,
    "category": "الحيض والاستحاضة والنفاس",
    "question": "كيف تميز المرأة بين دم الحيض ودم الاستحاضة؟"
  }
]

print(f"✓ Loaded {len(QUESTIONS_500):,} questions (IDs {QUESTIONS_500[0]['question_id']}..{QUESTIONS_500[-1]['question_id']}).")

df_q = pd.DataFrame(QUESTIONS_500)
print(f"\nBreakdown across {df_q['category'].nunique()} distinct categories:")
print(df_q['category'].value_counts().to_string())


In [ ]:
# 4. Load Qwen 0.6B Embedder and Mizan Reranker on GPU
from sentence_transformers import SentenceTransformer, CrossEncoder

EMBEDDER_NAME = "Qwen/Qwen3-Embedding-0.6B"
RERANKER_NAME = "ALJIACHI/Mizan-Rerank-V2"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"1. Loading embedder ({EMBEDDER_NAME}) on [{device}]...")
embedder = SentenceTransformer(EMBEDDER_NAME, device=device)
embedder.max_seq_length = 1024

print(f"2. Loading Mizan reranker ({RERANKER_NAME}) on [{device}]...")
reranker = CrossEncoder(RERANKER_NAME, max_length=1024, device=device, trust_remote_code=True)

# Pre-normalize corpus embeddings on GPU for fast dot product
print("3. Preparing GPU corpus index...")
corpus_embs_gpu = corpus_embeddings.to(device).float()
corpus_embs_gpu = torch.nn.functional.normalize(corpus_embs_gpu, p=2, dim=1)
print("✓ Models and index ready on GPU!")


In [ ]:
# 5. Execute 6,000 Candidate Generation (500 Questions x 12 Candidates/Q)
SCHOLAR_QUOTAS = {
    "Ibn Baz": 4,
    "Ibn Uthaymeen": 2,
    "Al-Lajnah Ad-Daimah": 2,
    "Saleh Al-Fawzan": 1,
    "Abdullah al-Manee": 1,
    "Abdulaziz Aal ash-Sheikh": 1,
    "Saleh bin Humaid": 1,
}

# Step A: Encode all 500 queries
queries_text = [q["question"] for q in QUESTIONS_500]
print(f"Encoding {len(queries_text)} queries on GPU with Qwen 0.6B...")
query_embs = embedder.encode(queries_text, batch_size=32, convert_to_tensor=True, normalize_embeddings=True, show_progress_bar=True)
query_embs = query_embs.to(corpus_embs_gpu.dtype)

# Step B: Compute similarity matrix (500 x 40,062)
print("Computing global vector similarities...")
similarity_matrix = torch.matmul(query_embs, corpus_embs_gpu.T).cpu().numpy()

# Pre-index corpus rows by scholar
scholar_indices = {}
for scholar in SCHOLAR_QUOTAS:
    scholar_indices[scholar] = np.where(corpus_df["source"].values == scholar)[0]

# Step C: Retrieve & Rerank 6,000 candidates
new_candidates_6k = []
candidate_id_counter = 3001  # IDs 1..3000 reserved for existing labeled items

print(f"\nRetrieving & Mizan-reranking candidate fatwas across 500 questions...")
t0 = time.time()

for q_idx, q_item in enumerate(tqdm(QUESTIONS_500, desc="Generating 6K Candidates")):
    q_id = q_item["question_id"]
    q_text = q_item["question"]
    q_cat = q_item["category"]
    
    q_sims = similarity_matrix[q_idx]
    
    for scholar, quota in SCHOLAR_QUOTAS.items():
        idx_pool = scholar_indices.get(scholar)
        if idx_pool is None or len(idx_pool) == 0:
            continue
            
        pool_size = min(len(idx_pool), max(quota * 3, 10))
        top_sim_subindices = np.argpartition(-q_sims[idx_pool], pool_size - 1)[:pool_size]
        top_corpus_indices = idx_pool[top_sim_subindices]
        
        pairs = []
        for c_idx in top_corpus_indices:
            row = corpus_df.iloc[c_idx]
            ans_snippet = str(row.get("answer_text", ""))[:1200]
            pairs.append([q_text, f"{row.get('title', '')}\n{ans_snippet}"])
            
        scores = reranker.predict(pairs, show_progress_bar=False, batch_size=32)
        sorted_order = np.argsort(-scores)
        
        for rank, rank_idx in enumerate(sorted_order[:quota], start=1):
            c_idx = top_corpus_indices[rank_idx]
            row = corpus_df.iloc[c_idx]
            r_score = float(scores[rank_idx])
            v_score = float(q_sims[c_idx])
            
            new_candidates_6k.append({
                "id": candidate_id_counter,
                "question_id": q_id,
                "category": q_cat,
                "question": q_text,
                "scholar": scholar,
                "fatwa_id": str(row.get("fatwa_id", "")),
                "title": str(row.get("title", "")),
                "question_text": str(row.get("question_text", "")),
                "answer_text": str(row.get("answer_text", "")),
                "reranker_score": r_score,
                "embedding_score": v_score,
                "rank_in_scholar": rank,
                "dataset_split": "new_6k_batch",
            })
            candidate_id_counter += 1

elapsed = time.time() - t0
print(f"\n✓ Successfully generated {len(new_candidates_6k):,} new candidate pairs in {elapsed/60:.2f} mins!")


In [ ]:
# 6. Ingest Existing Labeled 3,000 Candidates & Merge into 9,000 Pool
OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

existing_3k = []

# Search for the previously labeled dataset
search_sources = [
    Path("/kaggle/input/candidates_6k_bundle.zip"),
    Path("/kaggle/input/official_labeled_3k.json"),
    Path("/kaggle/input/official_labeled_3k.csv"),
    Path("/kaggle/working/candidates_6k_bundle.zip"),
    Path("/kaggle/working/official_labeled_3k.json"),
    Path("data/candidates_6k_bundle/official_labeled_3k.json"),
    Path("data/candidates_6k_bundle.zip"),
]
# Also search any uploaded datasets
for p in Path("/kaggle/input").rglob("*"):
    if p.name in ("official_labeled_3k.json", "candidates_6k_bundle.zip", "official_labeled_3k.csv"):
        search_sources.insert(0, p)

found_src = None
for s in search_sources:
    if s.exists():
        found_src = s
        break

assert found_src is not None, "Could not find existing labeled 3k dataset! Please attach candidates_6k_bundle.zip or official_labeled_3k.json under Input."
print(f"✓ Found existing 3k dataset at: {found_src}")

if found_src.name.endswith(".zip"):
    with zipfile.ZipFile(found_src, "r") as zf:
        if "official_labeled_3k.json" in zf.namelist():
            with zf.open("official_labeled_3k.json") as f:
                existing_3k = json.load(f)
        elif "official_labeled_3k.csv" in zf.namelist():
            with zf.open("official_labeled_3k.csv") as f:
                existing_3k = pd.read_csv(f).to_dict(orient="records")
elif found_src.name.endswith(".json"):
    with open(found_src, "r", encoding="utf-8") as f:
        existing_3k = json.load(f)
elif found_src.name.endswith(".csv"):
    existing_3k = pd.read_csv(found_src).to_dict(orient="records")

if isinstance(existing_3k, dict):
    existing_3k = list(existing_3k.values())

print(f"✓ Loaded {len(existing_3k):,} previously labeled items.")

# Merge existing 3,000 + new 6,000 = 9,000
dataset_9k = existing_3k + new_candidates_6k
print(f"\n🎯 Total Merged Dataset Size: {len(dataset_9k):,} items!")
print(f"  • Already Labeled: {len(existing_3k):,} items (Will NOT be re-billed)")
print(f"  • Awaiting Labels: {len(new_candidates_6k):,} items")


In [ ]:
# 7. Configure OpenRouter API Key
import os
import getpass

OPENROUTER_KEY = None

# Method 1: Kaggle Secrets (Add 'OPENROUTER_API_KEY' under Add-ons -> Secrets)
try:
    from kaggle_secrets import UserSecretsClient
    OPENROUTER_KEY = UserSecretsClient().get_secret("OPENROUTER_API_KEY")
    if OPENROUTER_KEY:
        print("✓ Loaded OpenRouter API key from Kaggle Secrets.")
except Exception:
    pass

# Method 2: Environment variable
if not OPENROUTER_KEY:
    OPENROUTER_KEY = os.getenv("OPENROUTER_API_KEY")
    if OPENROUTER_KEY:
        print("✓ Loaded OpenRouter API key from environment variable.")

# Method 3: Interactive prompt
if not OPENROUTER_KEY:
    OPENROUTER_KEY = getpass.getpass("🔑 Enter your OpenRouter API Key (input will be hidden): ").strip()

assert OPENROUTER_KEY and OPENROUTER_KEY.startswith("sk-"), "Invalid or missing OpenRouter API key!"
print("✓ OpenRouter API key verified.")


In [ ]:
# 8. Concurrent GPT-4o Mini Binary Labeling of the 6,000 New Items Only
import requests
import re
from concurrent.futures import ThreadPoolExecutor, as_completed

LABELED_OUTPUT_FILE = OUTPUT_DIR / "dataset_9k_labeled.json"
LABELED_CSV_FILE = OUTPUT_DIR / "dataset_9k_labeled.csv"

# Pre-populate labeled cache with the existing 3,000 items
labeled_cache = {str(it["id"]): it for it in existing_3k}

# If resuming a previous run of this cell, load any already saved 9k items
if LABELED_OUTPUT_FILE.exists():
    try:
        with open(LABELED_OUTPUT_FILE, "r", encoding="utf-8") as f:
            saved_dict = json.load(f)
            if isinstance(saved_dict, dict):
                labeled_cache.update(saved_dict)
            elif isinstance(saved_dict, list):
                for it in saved_dict:
                    labeled_cache[str(it["id"])] = it
        print(f"✓ Resuming checkpoint: {len(labeled_cache):,} items already labeled!")
    except Exception as e:
        print(f"⚠️ Checkpoint read warning: {e}")

def save_checkpoint(cache: dict):
    with open(LABELED_OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(cache, f, ensure_ascii=False, indent=2)

OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"

SYS_PROMPT = (
    "You are an expert Islamic fatwa relevance evaluation judge.\n"
    "Classify whether the provided Islamic fatwa directly and specifically answers the user's question.\n"
    "You must return ONLY ONE of the following two verdicts:\n"
    "1. 'answers-it': The fatwa directly and specifically answers the question asked.\n"
    "2. 'does-not-answer': The fatwa does NOT directly answer the question (including when it only touches on a related topic without directly answering, or is unrelated).\n\n"
    "Return a JSON object with:\n"
    "- 'verdict': exactly 'answers-it' or 'does-not-answer'\n"
    "- 'confidence': float between 0.0 and 1.0\n"
    "- 'explanation': brief explanation in Arabic."
)

def label_single_item(item: dict, max_retries: int = 3) -> dict:
    user_prompt = (
        f"User Question: {item['question']}\n\n"
        f"Scholar: {item['scholar']}\n"
        f"Title: {item['title']}\n"
        f"Fatwa Text: {item['answer_text'][:1400]}"
    )
    payload = {
        "model": "openai/gpt-4o-mini",
        "messages": [
            {"role": "system", "content": SYS_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        "temperature": 0.0,
        "response_format": {"type": "json_object"},
    }
    headers = {
        "Authorization": f"Bearer {OPENROUTER_KEY}",
        "Content-Type": "application/json",
        "HTTP-Referer": "https://kaggle.com",
        "X-Title": "Fatwa-RAG-Eval-9K",
    }
    
    last_err = None
    for attempt in range(max_retries):
        try:
            t0 = time.perf_counter()
            resp = requests.post(OPENROUTER_URL, headers=headers, json=payload, timeout=45)
            dt = time.perf_counter() - t0
            
            if resp.status_code == 429:
                time.sleep(2 * (attempt + 1))
                continue
            if resp.status_code >= 400:
                raise RuntimeError(f"API Error {resp.status_code}: {resp.text[:200]}")
            
            data = resp.json()
            content = data["choices"][0]["message"]["content"]
            usage = data.get("usage", {}) or {}
            
            verdict = "does-not-answer"
            conf = 1.0
            expl = ""
            try:
                m = re.search(r"\{.*?\}", content, re.DOTALL)
                if m:
                    parsed = json.loads(m.group(0))
                    raw_v = str(parsed.get("verdict", "")).lower().replace("_", "-").strip()
                    if raw_v in ("answers-it", "answers", "yes", "answers_it"):
                        verdict = "answers-it"
                    else:
                        verdict = "does-not-answer"
                    conf = float(parsed.get("confidence", 1.0))
                    expl = str(parsed.get("explanation", ""))
            except Exception:
                c_low = content.lower()
                if "answers-it" in c_low or "answers it" in c_low:
                    verdict = "answers-it"
                else:
                    verdict = "does-not-answer"
                
            return {
                "verdict": verdict,
                "confidence": conf,
                "explanation": expl,
                "cost": float(usage.get("cost", 0.0)),
                "latency_s": dt,
            }
        except Exception as e:
            last_err = e
            time.sleep(1.5 * (attempt + 1))
            
    raise RuntimeError(f"Failed after {max_retries} attempts: {last_err}")

# Filter items still needing labels (only the new 6,000 items that aren't cached yet)
pending_items = [it for it in dataset_9k if str(it["id"]) not in labeled_cache or "gpt_verdict" not in labeled_cache[str(it["id"])]]
print(f"Pending items to label: {len(pending_items):,} (Already labeled: {len(labeled_cache) - len(pending_items):,})")

MAX_WORKERS = 10
checkpoint_interval = 25
count_since_save = 0

if pending_items:
    pbar = tqdm(total=len(pending_items), desc="Labeling New 6K with GPT-4o Mini")
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        future_to_item = {executor.submit(label_single_item, it): it for it in pending_items}
        for future in as_completed(future_to_item):
            it = future_to_item[future]
            cid = str(it["id"])
            try:
                res = future.result()
                labeled_cache[cid] = {
                    **it,
                    "gpt_verdict": res["verdict"],
                    "gpt_confidence": res["confidence"],
                    "gpt_explanation": res["explanation"],
                    "cost": res["cost"],
                    "latency_s": res["latency_s"],
                }
                count_since_save += 1
                if count_since_save >= checkpoint_interval:
                    save_checkpoint(labeled_cache)
                    count_since_save = 0
            except Exception as e:
                print(f"\n[Warning] Item #{cid} error: {e}")
            pbar.update(1)
            
    save_checkpoint(labeled_cache)
    print(f"\n✓ Completed labeling! Checkpoint saved to {LABELED_OUTPUT_FILE.name}.")

# Export complete 9,000 dataset to CSV
df_9k = pd.DataFrame(list(labeled_cache.values()))
df_9k.to_csv(LABELED_CSV_FILE, index=False, encoding="utf-8")
print(f"✓ Saved complete 9,000 CSV dataset: {LABELED_CSV_FILE.name} ({len(df_9k):,} rows)")


In [ ]:
# 9. Diagnostics Report & 9k Dataset Archive Download
df_9k = pd.DataFrame(list(labeled_cache.values()))
total_items = len(df_9k)
answers_count = (df_9k["gpt_verdict"] == "answers-it").sum()
no_answers_count = (df_9k["gpt_verdict"] == "does-not-answer").sum()
total_cost = df_9k["cost"].sum()

print("=" * 80)
print("🎯 COMPLETE 9,000 DATASET BALANCE REPORT")
print("=" * 80)
print(f"Total Items in Dataset:              {total_items:,}")
print(f"Answers it ('answers-it'):            {answers_count:,} ({100*answers_count/total_items:.1f}%)")
print(f"Does not answer ('does-not-answer'):  {no_answers_count:,} ({100*no_answers_count/total_items:.1f}%)")
print(f"Total API Cost for new labeling:     ${total_cost:.4f}")
print("=" * 80)

print("\nVERDICT DISTRIBUTION BY SCHOLAR:")
scholar_summary = pd.crosstab(df_9k["scholar"], df_9k["gpt_verdict"], margins=True)
display(scholar_summary)

# Build complete 9,000 dataset ZIP archive for one-click download & Notebook 2
zip_path = OUTPUT_DIR / "dataset_9k_bundle.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(LABELED_CSV_FILE, arcname="dataset_9k_labeled.csv")
    zf.write(LABELED_OUTPUT_FILE, arcname="dataset_9k_labeled.json")

print("\n" + "=" * 80)
print(f"✓ All files archived to {zip_path.name} ({zip_path.stat().st_size / (1024**2):.2f} MB)")
print("You can download dataset_9k_bundle.zip directly from the Kaggle Output panel,")
print("and attach it to Notebook 2 to train and benchmark Laya!")
print("=" * 80)
